# PETS — planifier dans un modèle qui connaît ses limites

> **PETS** = *Probabilistic Ensembles with Trajectory Sampling* ([Chua, Calandra, McAllister, Levine,
> NeurIPS 2018, *« Deep RL in a Handful of Trials »*](https://proceedings.neurips.cc/paper/2018/hash/3de568f8597b94bda53149c7d7f5958c-Abstract.html)). C'est la **première grande méthode deep
> model-based orientée planning** : elle apprend un modèle de la dynamique avec un **ensemble de
> réseaux probabilistes**, puis **planifie en ligne** par optimisation (CEM) au lieu d'apprendre une
> politique. Elle atteint sur HalfCheetah des performances proches de SAC avec **10 à 100× moins
> d'interactions réelles**.

Ce notebook poursuit le fil model-based :

| Notebook | Méthode | Modèle du monde | Comment on agit |
|----------|---------|-----------------|-----------------|
| 10 — PILCO | analytique, ultra sample-efficient | **Gaussian Process** | politique RBF optimisée hors-ligne |
| 10b — Deep PILCO | passage à l'échelle | BNN (MC-dropout) | politique optimisée par particules |
| **11 — PETS** (ici) | ensembles + planning | **ensemble de réseaux probabilistes** | **MPC + CEM en ligne** |

PETS empile trois idées nouvelles, chacune introduite **comme un petit cours** : (1) séparer
l'incertitude **aléatoire** de l'incertitude **épistémique** ; (2) propager des **particules** à
travers l'ensemble (*trajectory sampling*, TS1 vs TS∞) ; (3) **planifier** par la méthode
d'entropie croisée (CEM). À la fin, vous devriez pouvoir réexpliquer *pourquoi* chaque brique existe.

## 1. Pourquoi PETS, après PILCO ?

PILCO est magnifique mais **ne passe pas à l'échelle**. Son Gaussian Process coûte $O(n^3)$ en
nombre de transitions et son *moment matching* analytique n'existe en forme fermée que pour des
noyaux et des politiques très particuliers. Sur un système à 18 dimensions comme HalfCheetah, avec
des dizaines de milliers de transitions, l'approche analytique s'effondre.

PETS remplace donc les deux piliers de PILCO par des objets qui **scalent** :

- le **GP** → un **ensemble de réseaux de neurones** (coût linéaire en données, gère la haute dimension) ;
- le **moment matching analytique** → une **propagation par particules** (échantillonnage, pas d'intégrale fermée) ;
- la **politique paramétrique optimisée hors-ligne** → un **planificateur en ligne** (MPC) qui, à
  chaque pas, cherche la meilleure séquence d'actions *dans le modèle*.

Mais en abandonnant le GP, on perd sa belle quantification d'incertitude. Le cœur de PETS est de la
**reconstruire avec des réseaux** — et pour cela il faut d'abord comprendre qu'il existe *deux
sortes* d'incertitude très différentes.

## 2. Les deux questions de PETS

Tout l'algorithme répond à deux questions, dans l'ordre :

1. **Comment représenter l'incertitude d'un modèle neuronal, de façon calibrée et scalable ?**
   → un **ensemble** de réseaux qui sortent chacun une *distribution* (moyenne + variance). On verra
   que cela capture séparément le bruit irréductible (aléatoire) et l'ignorance du modèle (épistémique).

2. **Comment se servir de ce modèle incertain pour bien agir ?**
   → on **planifie** : à chaque pas réel, on simule des milliers de futurs possibles à travers
   l'ensemble (*trajectory sampling*) et on optimise la séquence d'actions par **CEM**, puis on
   n'exécute que la première action et on recommence (MPC à horizon fuyant).

Les sections qui suivent construisent ces deux réponses brique par brique, en commençant par la plus
fondamentale : *qu'est-ce qu'on ne sait pas, exactement ?*

## 3. La boucle PETS en un coup d'œil

```mermaid
flowchart TB
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef world fill:none,stroke:#2563eb,stroke-width:2px
    classDef model fill:none,stroke:#d97706,stroke-width:2px
    classDef plan fill:none,stroke:#16a34a,stroke-width:2px

    collect["1. COLLECTER<br/>Agir dans le vrai monde<br/>Stocker $$~(s_t,a_t,s_{t+1})$$"]
    fit["2. AJUSTER<br/>Ré-entraîner l'ensemble probabiliste<br/>$$~p_\theta(s_{t+1}\mid s_t,a_t)$$"]
    plan["3. PLANIFIER<br/>À chaque pas réel, CEM cherche<br/>la meilleure séquence d'actions dans le modèle"]
    act["4. AGIR<br/>Exécuter seulement la première action<br/>Observer le nouvel état, puis replanifier"]

    collect -->|"dataset de transitions"| fit
    fit -->|"modèle mis à jour"| plan
    plan -->|"première action du plan MPC"| act
    act -->|"nouvelle transition réelle"| collect

    class collect,act world
    class fit model
    class plan plan
```

| Quick-fact | PETS |
|---|---|
| Type | model-based, planning (pas de politique apprise) |
| Modèle | ensemble de $B$ réseaux probabilistes (mean + variance) |
| Incertitude | aléatoire (têtes de variance) **+** épistémique (désaccord d'ensemble) |
| Propagation | particules, *trajectory sampling* (TS1 / TS∞) |
| Optimiseur d'actions | CEM (cross-entropy method), horizon fuyant |
| Environnement (ici) | HalfCheetah-v5 (obs 18-dim, action 6-dim ∈ [−1,1]) |
| Hypothèse | fonction de récompense **connue** sur l'état |

PETS sépare clairement le vrai monde et le monde simulé. Les transitions réelles servent à
réajuster le modèle ; le modèle sert ensuite à tester des milliers de plans candidats avec CEM ; mais
une seule action est réellement exécutée avant de revenir à l'observation. C'est le principe MPC :
planifier loin, agir court, corriger souvent.

In [ ]:
import time
from pathlib import Path

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Video, display

# On reste en float32 (réseaux NN) — contrairement à PILCO qui tournait en float64
# pour la précision des GPs. Les NNs sont stables en float32 et c'est beaucoup plus rapide.
torch.set_default_dtype(torch.float32)


plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

print("Environnement prêt — torch", torch.__version__)


---
# Partie I — Deux incertitudes : aléatoire vs épistémique

## I.1 — Le dé et le mot étranger

Avant toute formule, il faut sentir qu'« être incertain » recouvre **deux choses radicalement
différentes**.

**L'incertitude aléatoire** est celle d'un **dé**. Même si vous connaissez parfaitement le dé, sa
physique, l'état du monde — le résultat reste imprévisible. C'est un **bruit irréductible**, inhérent
au système. Collecter plus de données *ne la réduit pas* : un dé observé un million de fois reste un
dé. Sur un robot, c'est le bruit des capteurs, les frottements stochastiques, un contact qui glisse.

**L'incertitude épistémique** est celle d'un **mot étranger que vous n'avez jamais appris**. Vous
hésitez non pas parce que le mot est aléatoire, mais parce qu'il vous **manque de la connaissance**.
Dès que quelqu'un vous l'explique, l'hésitation disparaît. C'est l'ignorance du *modèle*, et elle
**fond avec les données** : dans les régions où le robot a déjà bougé, le modèle sait ; là où il n'a
jamais mis les pieds, il devine.

Pourquoi cette distinction est-elle vitale pour PETS ? Parce qu'un bon agent **explore** là où
l'incertitude est *épistémique* (il peut apprendre quelque chose) et reste **prudent** face à
l'incertitude *aléatoire* (rien à apprendre, juste du bruit à moyenner). Confondre les deux, c'est
soit explorer du bruit pour rien, soit construire un plan confiant sur une région que le modèle n'a
jamais vue.

## I.2 — Les deux incertitudes, formellement

On modélise la dynamique par une distribution conditionnelle $p(s' \mid s, a)$ : à partir d'un état
$s$ et d'une action $a$, le modèle ne prédit pas seulement **un** prochain état, mais une
**distribution** de prochains états possibles.

Pour simplifier la notation, imaginons d'abord une seule dimension de $s'$. Un modèle de paramètres
$\theta$ prédit :

$$s' \mid s,a,\theta \sim \mathcal{N}\!\big(\mu_\theta(s,a),\, \sigma_\theta^2(s,a)\big)$$

Ici, la variance $\sigma_\theta^2(s,a)$ ne tombe pas du ciel : elle est **prédite par le réseau**
lui-même, via une tête de variance. Elle représente l'incertitude que le modèle attribue au monde
même si ses paramètres étaient connus. Par exemple : contacts physiques bruités, transitions
difficiles à prédire, ou plusieurs prochains états plausibles pour une même entrée.

Mais nous sommes aussi incertains sur les paramètres $\theta$, car le modèle est appris avec un
dataset fini $\mathcal{D}$. En théorie bayésienne, on devrait donc raisonner avec une distribution
sur les modèles plausibles : $p(\theta \mid \mathcal{D})$. En pratique, PETS approxime cette
distribution avec un **ensemble** de réseaux.

La loi des variances totales donne alors :

$$\underbrace{\operatorname{Var}[s']}_{\text{incertitude totale}} =
\underbrace{\mathbb{E}_{\theta}\big[\sigma_\theta^2(s,a)\big]}_{\textbf{incertitude aléatoire}} \;+\;
\underbrace{\operatorname{Var}_{\theta}\big[\mu_\theta(s,a)\big]}_{\textbf{incertitude épistémique}}$$

| Terme | Lecture |
|-------|---------|
| $\sigma_\theta^2(s,a)$ | le bruit que **chaque** modèle déclare lui-même pour cette transition |
| $\mathbb{E}_\theta[\sigma_\theta^2]$ | **aléatoire** : le bruit moyen déclaré par les modèles plausibles |
| $\mu_\theta(s,a)$ | la prédiction moyenne d'**un** modèle donné |
| $\operatorname{Var}_\theta[\mu_\theta]$ | **épistémique** : à quel point les modèles plausibles *se contredisent* |

Donc il y a deux questions différentes :

- **Même avec le bon modèle, le monde est-il intrinsèquement bruité ?**  
  C'est l'incertitude aléatoire.

- **Les modèles appris ne sont-ils pas d'accord parce qu'on manque de données ?**  
  C'est l'incertitude épistémique.

PETS rend ces deux termes concrets : l'aléatoire vient de la **tête de variance** de chaque réseau
(Partie II), l'épistémique vient du **désaccord entre les membres** d'un ensemble (Partie III).
On vérifiera numériquement cette identité plus loin.

In [ ]:
# Jeu de données jouet 1D hétéroscédastique avec un trou
# f(x) = sin(1.5*x), bruit croissant avec x (côté droit plus bruité)

np.random.seed(42)

def f_true(x):
    return np.sin(1.5 * x)

def noise_std(x):
    # Bruit hétéroscédastique : faible à gauche, croissant à droite
    return 0.05 + 0.20 * np.maximum(x, 0.0)

# Deux intervalles : [-3,-1] et [1,3] — trou au centre (-1,1)
x_left  = np.random.uniform(-3.0, -1.0, 100).astype(np.float32)
x_right = np.random.uniform( 1.0,  3.0, 100).astype(np.float32)
x_train = np.concatenate([x_left, x_right])
y_train = f_true(x_train) + noise_std(x_train) * np.random.randn(len(x_train)).astype(np.float32)
y_train = y_train.astype(np.float32)

# Grille fine pour les visualisations
x_grid = np.linspace(-3.5, 3.5, 300).astype(np.float32)

# Visualisation
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_train, y_train, s=10, alpha=0.5, color="steelblue", label="Données")
ax.plot(x_grid, f_true(x_grid), "k--", lw=2, label="f vraie")
ax.axvspan(-1, 1, alpha=0.10, color="orange", label="Trou (pas de données)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Jouet 1D : bruit hétéroscédastique avec trou central")
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Deux « inconnues » coexistent sur ce graphe et n'ont rien à voir. À **droite**, les
points sont très dispersés autour de la courbe : c'est de l'**aléatoire** — le bruit grandit avec
$x$, et aucune quantité de données ne le supprimera. Au **centre** (l'intervalle $(-1,1)$), il n'y a
**aucun point** : c'est de l'**épistémique** — on ne sait pas, faute d'avoir observé. Un bon modèle
doit dire « ça varie beaucoup ici » à droite **et** « je ne sais pas ici » au centre — et surtout, ne
pas confondre les deux. C'est exactement ce que PETS va réussir.

## I.3 — Pourquoi un planificateur a besoin des deux

Imaginez l'agent qui planifie une séquence d'actions en imaginant le futur dans son modèle. Deux
pièges symétriques le guettent :

- **Ignorer l'épistémique** = bâtir un plan sur du sable. Le modèle extrapole avec assurance dans une
  région jamais visitée, le plan a l'air excellent *dans l'imagination*, et se vautre dans la réalité.
  C'est le « pilote dans le brouillard » de PILCO : sans conscience de son ignorance, il fonce.
- **Ignorer l'aléatoire** = se laisser piéger par la chance. Une séquence d'actions peut sembler
  géniale parce qu'un tirage de bruit favorable l'a flattée. En moyennant sur **plusieurs particules**
  (plusieurs tirages du bruit aléatoire), PETS évalue le rendement *attendu*, pas un coup de chance.

D'où l'architecture : l'ensemble fournit l'épistémique (désaccord), les têtes de variance fournissent
l'aléatoire, et la propagation par particules **moyenne** sur les deux pour scorer honnêtement un plan.

Pour capturer l'aléatoire, notre réseau ne peut plus se contenter de prédire **un point** $\hat s'$.
Il doit prédire une **distribution** — au minimum une moyenne *et* une variance. C'est le réseau
probabiliste, brique de la Partie II.

---
# Partie II — Le réseau probabiliste

## II.1 — Sortir une distribution, pas un point

Un réseau de régression classique apprend $x \mapsto \hat y$ : pour une entrée donnée, il donne une
valeur moyenne, puis se tait sur sa confiance. C'est suffisant si l'on veut seulement prédire un
point, mais insuffisant pour PETS : le planificateur doit savoir **quand le modèle est fiable** et
quand il doit être prudent.

Un **réseau probabiliste** apprend plutôt :

$$x \mapsto \big(\mu(x),\, \sigma^2(x)\big)$$

Il sort donc deux têtes :

- une tête de **moyenne** $\mu(x)$ : la meilleure prédiction centrale ;
- une tête de **variance** $\sigma^2(x)$ : l'incertitude aléatoire autour de cette prédiction.

On suppose alors localement :

$$y \mid x \sim \mathcal{N}\!\big(\mu(x),\, \sigma^2(x)\big)$$

Le point important est que $\sigma^2(x)$ peut **dépendre de l'entrée**. On dit que le bruit est
**hétéroscédastique**. Le mot est technique, mais l'idée est simple : le monde n'est pas forcément
aussi prévisible partout.

Par exemple, en dynamique :

- dans une zone stable, loin des contacts violents, deux transitions proches se ressemblent beaucoup :
  le modèle peut prédire avec une petite variance ;
- dans une zone instable, proche d'un contact, d'une chute ou d'une transition mal couverte par les
  données, la même action peut produire des effets plus variables :
  le modèle doit annoncer une variance plus grande.

Intuitivement, l'hétéroscédasticité signifie donc : **la largeur de la cloche dépend de l'endroit où
l'on se trouve**. Ce n'est pas une unique barre d'erreur globale collée à tout le dataset ; c'est une
barre d'erreur locale, adaptée à chaque entrée.

C'est exactement ce qu'on veut dans PETS : le planificateur ne doit pas seulement demander
« quelle trajectoire a la meilleure récompense prédite ? », mais aussi « est-ce que cette trajectoire
passe par des régions où le modèle avoue beaucoup de bruit ? ».

Reste à choisir une fonction de perte qui force la variance à être honnête. La MSE ne suffit pas :
elle entraîne seulement la moyenne et ignore $\sigma^2(x)$. La bonne perte est la
**vraisemblance gaussienne négative**, qui récompense une bonne moyenne, mais pénalise aussi une
variance trop petite ou trop grande.

In [ ]:
# Illustration : bruit homoscédastique vs hétéroscédastique
# - Homoscédastique : même niveau de bruit partout.
# - Hétéroscédastique : le bruit dépend de x.

rng = np.random.default_rng(0)
x = np.linspace(-3, 3, 220)

# Même signal moyen dans les deux cas.
true_mean = np.sin(1.5 * x)

# Cas 1 : variance constante.
sigma_homo = 0.25 * np.ones_like(x)
y_homo = true_mean + rng.normal(0, sigma_homo)

# Cas 2 : variance dépendante de x.
# Ici le monde devient plus bruité à droite.
sigma_hetero = 0.12 + 0.55 / (1.0 + np.exp(-2.0 * (x - 0.4)))
y_hetero = true_mean + rng.normal(0, sigma_hetero)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)

for ax, y, sigma, title in [
    (axes[0], y_homo, sigma_homo, "Bruit homoscédastique\nmême variance partout"),
    (axes[1], y_hetero, sigma_hetero, "Bruit hétéroscédastique\nvariance dépendante de x"),
]:
    ax.scatter(x, y, s=14, alpha=0.55, label="observations bruitées")
    ax.plot(x, true_mean, color="black", lw=2, label="moyenne vraie")

    # Bande ±2σ : elle montre visuellement la variance locale.
    ax.fill_between(
        x,
        true_mean - 2 * sigma,
        true_mean + 2 * sigma,
        color="tab:orange",
        alpha=0.22,
        label=r"zone $\mu(x) \pm 2\sigma(x)$",
    )

    ax.set_title(title)
    ax.set_xlabel("entrée x")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("sortie y")
axes[0].legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

Lecture. À gauche, la largeur de la bande orange est constante : le modèle pourrait utiliser une seule variance globale. À droite, la bande s’élargit quand $x$ augmente : un réseau probabiliste doit pouvoir prédire une variance locale $\sigma^2(x)$, pas seulement une moyenne.

## II.2 — La vraisemblance gaussienne négative (NLL), terme par terme

On veut maximiser la vraisemblance des données sous la gaussienne prédite, donc **minimiser** son
log-négatif. Pour une sortie de dimension $D$ avec covariance diagonale $\sigma^2_d(x)$ :

$$\mathcal{L}(x,y) = \frac{1}{2}\sum_{d=1}^{D}\left[
\frac{\big(y_d - \mu_d(x)\big)^2}{\sigma_d^2(x)} \;+\; \log \sigma_d^2(x) \right] \;+\; \text{const}$$

| Terme | Rôle |
|-------|------|
| $\dfrac{(y_d-\mu_d)^2}{\sigma_d^2}$ | **attache aux données pondérée par la confiance** : une erreur sur une dimension que le réseau dit *fiable* (σ petit) coûte cher ; sur une dimension qu'il dit *bruitée* (σ grand), elle coûte moins |
| $\log \sigma_d^2$ | **pénalité d'incertitude** : empêche la triche « je déclare σ énorme partout pour annuler le 1ᵉʳ terme » |
| const $= \frac{D}{2}\log 2\pi$ | indépendante des paramètres, on l'omet |

**L'équilibre est tout.** Le premier terme pousse $\sigma$ vers le haut (pour amortir les grosses
erreurs), le second le pousse vers le bas. À l'optimum, $\sigma_d^2(x)$ s'aligne sur la **variance
réelle des résidus** en $x$ : le réseau apprend *tout seul* à quel point il est bruité, région par
région. C'est ça, capturer l'aléatoire.

### — Le truc du `logvar` et les bornes douces

Deux problèmes pratiques. **(1)** $\sigma^2$ doit être strictement positif ; on ne le prédit donc pas
directement mais via son logarithme $v = \log \sigma^2$ (le `logvar`), libre dans $\mathbb{R}$, et on
récupère $$\sigma^2 = e^{v}$$. La NLL devient $$\frac{1}{2}\sum_d\big[(y_d-\mu_d)^2 e^{-v_d} + v_d\big]$$,
parfaitement lisse. **(2)** En début d'entraînement, le `logvar` peut exploser ou s'effondrer et
déstabiliser tout. PETS le **borne en douceur** entre deux paramètres *apprenables* `max_logvar` et
`min_logvar` via la softplus :

$$v \leftarrow v_{\max} - \operatorname{softplus}(v_{\max} - v), \qquad
v \leftarrow v_{\min} + \operatorname{softplus}(v - v_{\min})$$

| Étape | Effet |
|-------|-------|
| $v_{\max}-\operatorname{softplus}(v_{\max}-v)$ | plafonne $v$ : au-dessus de $v_{\max}$, ça sature en douceur (pas de coupure brutale) |
| $v_{\min}+\operatorname{softplus}(v-v_{\min})$ | plancher symétrique |
| régularisation $0.01\,(\sum v_{\max} - \sum v_{\min})$ | resserre légèrement les bornes apprises, pour qu'elles ne s'écartent pas inutilement |

C'est un détail d'implémentation, mais c'est *le* détail qui rend l'entraînement d'un réseau
probabiliste stable. Sans lui, la NLL diverge une fois sur deux.

### — Pourquoi pas une simple MSE ?

On pourrait entraîner deux têtes séparément : une MSE pour la moyenne, une autre régression pour la
variance. Mauvaise idée. La MSE seule apprend la moyenne **sans jamais regarder le bruit** : elle
traite une zone calme et une zone très bruitée de la même façon, et n'a aucun mécanisme pour exprimer
« ici je suis moins sûr ». La NLL **couple** moyenne et variance dans une seule perte cohérente : la
variance modère l'attache aux données là où c'est bruité, et la moyenne se concentre là où c'est
franc. C'est une seule descente de gradient, un seul objectif, deux sorties qui se calibrent ensemble.

In [ ]:
# ============================================================
# NLL gaussienne et réseau probabiliste (from scratch)
# ============================================================

def gaussian_nll(mean, logvar, target):
    # NLL diagonale-gaussienne : 0.5 * mean_batch[ sum_dim[ (y-mu)^2*exp(-logvar) + logvar ] ]
    # La constante 0.5*log(2*pi) est omise (ne dépend pas des paramètres)
    return 0.5 * ((target - mean) ** 2 * torch.exp(-logvar) + logvar).sum(-1).mean()


class ProbabilisticMLP(nn.Module):
    # Réseau probabiliste : prédit (mu, logvar) pour une gaussienne diagonale
    # Architecture : tronc Linear+SiLU x n_layers, puis tête Linear(2*output_dim)
    # Bornes douces sur logvar via softplus (stabilité d'entraînement)

    def __init__(self, input_dim, output_dim, hidden_dim=64, n_layers=2):
        super().__init__()
        self.output_dim = output_dim

        # Tronc : n_layers couches cachées
        layers = []
        in_features = input_dim
        for _ in range(n_layers):
            layers.append(nn.Linear(in_features, hidden_dim))
            layers.append(nn.SiLU())
            in_features = hidden_dim
        # Tête : 2*output_dim sorties (mean + raw_logvar)
        layers.append(nn.Linear(in_features, 2 * output_dim))
        self.net = nn.Sequential(*layers)

        # Bornes apprenables sur le logvar (initialisées comme dans le papier PETS)
        self.max_logvar = nn.Parameter(torch.full((output_dim,), 0.5))
        self.min_logvar = nn.Parameter(torch.full((output_dim,), -10.0))

    def forward(self, x):
        out = self.net(x)
        mean, raw_logvar = out.chunk(2, dim=-1)

        # Borne haute : max_logvar - softplus(max_logvar - raw_logvar)
        logvar = self.max_logvar - F.softplus(self.max_logvar - raw_logvar)
        # Borne basse : min_logvar + softplus(logvar - min_logvar)
        logvar = self.min_logvar + F.softplus(logvar - self.min_logvar)

        return mean, logvar


print("ProbabilisticMLP défini")

In [ ]:
# Mini-test visuel : gaussian_nll + ProbabilisticMLP shapes + bornes logvar

torch.manual_seed(0)

# (1) gaussian_nll vs calcul manuel sur un exemple contrôlé.
mean_t = torch.tensor([[1.0, 2.0]])
logvar_t = torch.tensor([[0.0, 1.0]])   # sigma^2 = [1, e]
target_t = torch.tensor([[1.5, 1.0]])

nll_auto = gaussian_nll(mean_t, logvar_t, target_t)
nll_manual = 0.5 * (
    ((1.5 - 1.0) ** 2 * 1.0 + 0.0)
    + ((1.0 - 2.0) ** 2 / 2.718281828 + 1.0)
)
assert abs(float(nll_auto) - nll_manual) < 1e-4, f"NLL mismatch: {nll_auto} vs {nll_manual}"

# (2) forward d'un ProbabilisticMLP(1,1) — shapes.
net = ProbabilisticMLP(1, 1, hidden_dim=16, n_layers=2)
x_dummy = torch.linspace(-2, 2, 80).reshape(-1, 1)
mu_out, lv_out = net(x_dummy)

assert mu_out.shape == (80, 1), f"mean shape: {mu_out.shape}"
assert lv_out.shape == (80, 1), f"logvar shape: {lv_out.shape}"

# (3) logvar respecte les bornes douces.
eps = 1e-3
assert (lv_out <= net.max_logvar.item() + eps).all(), "logvar dépasse max_logvar"
assert (lv_out >= net.min_logvar.item() - eps).all(), "logvar en dessous de min_logvar"

# Visualisation : le réseau non entraîné sort déjà deux têtes bien formées.
x_np = x_dummy.squeeze().detach().numpy()
mu_np = mu_out.squeeze().detach().numpy()
logvar_np = lv_out.squeeze().detach().numpy()
std_np = np.sqrt(np.exp(logvar_np))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].plot(x_np, mu_np, lw=2, label=r"$\mu_\theta(x)$")
axes[0].fill_between(
    x_np,
    mu_np - 2 * std_np,
    mu_np + 2 * std_np,
    alpha=0.25,
    label=r"$\mu_\theta(x) \pm 2\sigma_\theta(x)$",
)
axes[0].set_title("Sortie probabiliste du réseau")
axes[0].set_xlabel("x")
axes[0].set_ylabel("prédiction")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(x_np, logvar_np, lw=2, color="tab:orange", label=r"$\log \sigma^2_\theta(x)$")
axes[1].axhline(net.max_logvar.item(), color="tab:red", ls="--", label="max_logvar")
axes[1].axhline(net.min_logvar.item(), color="tab:blue", ls="--", label="min_logvar")
axes[1].set_title("La tête log-variance reste bornée")
axes[1].set_xlabel("x")
axes[1].set_ylabel("log-variance")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print(f"NLL auto     : {float(nll_auto):.6f}")
print(f"NLL manuelle : {nll_manual:.6f}")
print(f"Shapes       : mean={tuple(mu_out.shape)}, logvar={tuple(lv_out.shape)}")
print(
    "Logvar       : "
    f"min={logvar_np.min():.3f}, max={logvar_np.max():.3f} "
    f"(bornes [{net.min_logvar.item():.1f}, {net.max_logvar.item():.1f}])"
)

**Lecture**. Le mini-test ne vérifie pas encore que le réseau apprend : il vérifie que la brique est saine. La NLL correspond au calcul manuel, les deux têtes ont les bonnes dimensions, et la log-variance reste dans les bornes prévues. Le plot rend visible l’idée clé : le modèle ne sort pas seulement une courbe moyenne, mais une bande d’incertitude locale.

In [ ]:
# Entraîner un ProbabilisticMLP sur le jouet 1D
# et visualiser mean ± 2*sigma (incertitude aléatoire)

torch.manual_seed(0)
np.random.seed(0)

net1 = ProbabilisticMLP(1, 1, hidden_dim=64, n_layers=2)
optimizer = torch.optim.Adam(net1.parameters(), lr=1e-3)

X_tr = torch.from_numpy(x_train).unsqueeze(1)   # [N,1]
Y_tr = torch.from_numpy(y_train).unsqueeze(1)   # [N,1]

for step in range(1800):
    optimizer.zero_grad()
    mu, lv = net1(X_tr)
    # Perte NLL + régularisation légère sur les bornes
    loss = gaussian_nll(mu, lv, Y_tr) + 0.01 * (net1.max_logvar.sum() - net1.min_logvar.sum())
    loss.backward()
    optimizer.step()

net1.eval()
with torch.no_grad():
    x_t = torch.from_numpy(x_grid).unsqueeze(1)
    mu_g, lv_g = net1(x_t)
    mu_g  = mu_g.squeeze().numpy()
    std_g = torch.exp(0.5 * lv_g).squeeze().numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_train, y_train, s=10, alpha=0.4, color="steelblue", label="Données")
ax.plot(x_grid, f_true(x_grid), "k--", lw=2, label="f vraie")
ax.plot(x_grid, mu_g, color="crimson", lw=2, label="Moyenne prédite")
ax.fill_between(x_grid, mu_g - 2*std_g, mu_g + 2*std_g,
                alpha=0.25, color="crimson", label=r"$\pm 2\sigma$ aléatoire")
ax.axvspan(-1, 1, alpha=0.08, color="orange", label="Trou")
ax.set_title("Réseau probabiliste : bande aléatoire apprise (NLL)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lecture.** La bande $\pm 2\sigma$ **épouse le bruit** : étroite à gauche où les données sont franches,
large à droite où elles sont dispersées. Le réseau a appris l'**aléatoire** hétéroscédastique tout
seul, juste en minimisant la NLL — personne ne lui a dit où était le bruit. Mais regardez le **trou**
central : que vaut la bande là où il n'y a aucune donnée ?

In [ ]:
# Zoom sur le trou : montrer la surconfiance du réseau seul

# Calculer std moyen dans le trou vs dans les données
mask_trou  = (x_grid >= -1.0) & (x_grid <= 1.0)
mask_data  = ~mask_trou
std_trou   = float(std_g[mask_trou].mean())
std_data   = float(std_g[mask_data].mean())

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_train, y_train, s=10, alpha=0.4, color="steelblue", label="Données")
ax.plot(x_grid, f_true(x_grid), "k--", lw=2, label="f vraie")
ax.plot(x_grid, mu_g, color="crimson", lw=2, label="Moyenne prédite")
ax.fill_between(x_grid, mu_g - 2*std_g, mu_g + 2*std_g,
                alpha=0.25, color="crimson", label=r"$\pm 2\sigma$")
# Surligner le trou
ax.axvspan(-1, 1, alpha=0.15, color="orange", label="Trou (pas de données)")
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.0, 3.0)
ax.set_title(f"Surconfiance dans le trou : σ_trou={std_trou:.3f}  vs  σ_données={std_data:.3f}")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(fontsize=8)
ax.annotate("Confiant\nmais faux !", xy=(0, float(mu_g[len(mu_g)//2])),
            xytext=(0.5, 2.0), arrowprops=dict(arrowstyle="->"), fontsize=10, color="darkorange")
plt.tight_layout()
plt.show()

**Lecture.** Catastrophe silencieuse : dans le trou, le réseau est **confiant et faux**. Sa tête de
variance ne capture que l'aléatoire *observé près des données* ; elle n'a aucun moyen de dire « je
n'ai jamais rien vu ici ». Un seul réseau probabiliste **ne voit pas son propre trou de connaissance**
— il lui manque l'incertitude **épistémique**. La solution de PETS : ne pas faire confiance à un seul
réseau, mais à un **comité** de réseaux et écouter leur **désaccord**. C'est la Partie III.

---
# Partie III — Ensembles profonds & incertitude épistémique

## III.1 — Le comité d'experts

Reprenons l'analogie du **comité d'experts** (déjà croisée avec Deep PILCO). Au lieu de demander à
*un* expert, on en consulte $B$, formés indépendamment. Chaque membre de l'ensemble a la même
architecture, mais pas exactement la même histoire : initialisation différente, ordre des minibatches
différent, et parfois données légèrement rééchantillonnées (*bootstrap*).

L'idée n'est pas que chaque expert soit parfait. Au contraire : on exploite leurs différences.

Là où les données sont abondantes, le problème est bien contraint. Même avec des initialisations
différentes, tous les réseaux voient assez d'exemples pour apprendre une dynamique similaire :
leurs prédictions se regroupent. Ils **sont d'accord**.

Là où les données manquent, il n'existe pas encore assez d'évidence pour trancher. Plusieurs
fonctions peuvent expliquer le dataset observé, mais produire des extrapolations très différentes.
Chaque réseau choisit alors une hypothèse plausible, influencée par son initialisation et son
échantillon d'entraînement. Ils **divergent**.

C'est exactement le signal qui manquait. Le **désaccord entre membres** dans une région devient une
mesure pratique de l'incertitude épistémique :

> plusieurs modèles compatibles avec les données prédisent des futurs différents,
> donc le système ne sait pas encore vraiment ce qui va se passer.

Mathématiquement, si les membres prédisent des moyennes
$\mu_1(s,a), \dots, \mu_B(s,a)$, on peut estimer l'incertitude épistémique par la variance entre ces
moyennes :

$$\operatorname{Var}_{\text{ensemble}}\!\big[\mu_b(s,a)\big]$$

C'est une approximation simple de la variance sur les paramètres $\theta$ vue en Partie I.2. Elle ne
remplace pas une vraie inférence bayésienne complète, mais elle donne un signal robuste, facile à
implémenter, et très utile pour planifier.

Dans PETS, cette idée est cruciale : le planificateur ne suit pas aveuglément une trajectoire parce
qu'un seul modèle la trouve prometteuse. Il propage plusieurs particules à travers plusieurs membres
de l'ensemble. Si une trajectoire traverse une zone mal connue, les prédictions se dispersent, et
cette dispersion apparaît dans les rollouts imaginés.

C'est l'idée des *deep ensembles*
([Lakshminarayanan et al., 2017](https://arxiv.org/abs/1612.01474)), que PETS adopte : obtenir une
incertitude épistémique utile avec une mécanique très simple, sans écrire explicitement
$p(\theta \mid \mathcal{D})$.

## III.2 — D'où vient le désaccord : initialisations + bootstrap

Pour que les membres divergent *là où il faut* (et seulement là), il faut leur donner des raisons
raisonnables de ne pas apprendre exactement la même fonction. PETS utilise deux sources de diversité
simples.

1. **Initialisations aléatoires différentes.** Chaque réseau démarre d'un point distinct dans
   l'espace des poids. Là où les données sont nombreuses et cohérentes, la descente de gradient
   ramène les modèles vers des prédictions proches : le dataset impose fortement la solution. Mais
   loin des données, il existe beaucoup de fonctions compatibles avec ce qu'on a observé. Les réseaux
   gardent alors des extrapolations différentes.

2. **Bootstrap des données.** Chaque membre s'entraîne sur un ré-échantillonnage *avec remise* du jeu
   de données : certains points sont vus plusieurs fois, d'autres pas du tout. Cela imite une question
   statistique très utile : *quelle aurait été ma prédiction si mon dataset avait été légèrement
   différent ?* Si cette petite variation du dataset change fortement la prédiction, c'est le signe
   que la région est mal contrainte.

Ces deux mécanismes créent une diversité contrôlée. On ne veut pas du désaccord partout : si les
membres se contredisent même près des données, l'ensemble est juste mal entraîné. On veut un
comportement précis :

- **près des données** : les membres convergent, car ils ont tous vu assez d'évidence ;
- **entre les données ou loin du dataset** : les membres s'écartent, car l'extrapolation n'est pas
  déterminée par les observations.

Le résultat est un éventail de prédictions resserré près des données, puis évasé dans les trous.
**L'évasement est l'épistémique.** C'est une incertitude qui dit : *je pourrais apprendre cette zone
si je collectais plus de transitions ici*. On va le voir littéralement.

In [ ]:
# ============================================================
# Ensemble probabiliste : B membres entraînés sur bootstrap
# ============================================================

class ProbabilisticEnsemble(nn.Module):
    # Ensemble de B réseaux probabilistes indépendants
    # Chaque membre s'entraîne sur un bootstrap (tirage avec remise) du dataset
    # → diversité dans les zones sans données = incertitude épistémique

    def __init__(self, input_dim, output_dim, n_members=5, hidden_dim=64, n_layers=2):
        super().__init__()
        self.n_members  = n_members
        self.output_dim = output_dim
        self.members = nn.ModuleList([
            ProbabilisticMLP(input_dim, output_dim, hidden_dim=hidden_dim, n_layers=n_layers)
            for _ in range(n_members)
        ])

    def fit(self, X, Y, steps=1500, lr=1e-3, batch_size=64):
        # Entraîne chaque membre indépendamment sur son propre bootstrap
        N = X.shape[0]
        for b, member in enumerate(self.members):
            # Bootstrap : N indices avec remise
            idx = torch.randint(0, N, (N,))
            Xb, Yb = X[idx], Y[idx]
            opt = torch.optim.Adam(member.parameters(), lr=lr)
            member.train()
            for _ in range(steps):
                batch_idx = torch.randint(0, N, (min(batch_size, N),))
                xb, yb = Xb[batch_idx], Yb[batch_idx]
                opt.zero_grad()
                mu, lv = member(xb)
                loss = gaussian_nll(mu, lv, yb) + 0.01 * (member.max_logvar.sum() - member.min_logvar.sum())
                loss.backward()
                opt.step()
            member.eval()

    @torch.no_grad()
    def predict(self, x):
        # Retourne means [B, N_pts, out] et vars [B, N_pts, out]
        all_means, all_vars = [], []
        for member in self.members:
            mu, lv = member(x)
            all_means.append(mu.unsqueeze(0))
            all_vars.append(torch.exp(lv).unsqueeze(0))
        means = torch.cat(all_means, dim=0)  # [B, N_pts, out]
        vars_ = torch.cat(all_vars,  dim=0)  # [B, N_pts, out]
        return means, vars_


# Entraîner l'ensemble sur le jouet 1D
torch.manual_seed(1)
np.random.seed(1)

ensemble1d = ProbabilisticEnsemble(1, 1, n_members=5, hidden_dim=64, n_layers=2)
ensemble1d.fit(X_tr, Y_tr, steps=1500, lr=1e-3, batch_size=64)
print(f"Ensemble entraîné : {ensemble1d.n_members} membres")

In [ ]:
# Mini-test : shapes predict, NLL baisée, membres divergent dans le trou

x_test_dummy = torch.randn(10, 1)
means_t, vars_t = ensemble1d.predict(x_test_dummy)

assert means_t.shape == (5, 10, 1), f"shapes means: {means_t.shape}"
assert vars_t.shape  == (5, 10, 1), f"shapes vars:  {vars_t.shape}"

# Deux membres donnent des moyennes différentes dans le trou
x_trou = torch.zeros(1, 1)  # x=0 est dans le trou
m_trou, _ = ensemble1d.predict(x_trou)
m0 = float(m_trou[0, 0, 0])
m1 = float(m_trou[1, 0, 0])
assert abs(m0 - m1) > 0.05, f"Membres trop proches dans le trou : {m0:.3f} vs {m1:.3f}"

print(f"✓ shapes (5,10,1) OK — désaccord dans le trou: membre0={m0:.3f}, membre1={m1:.3f}")

In [ ]:
# Visualisation de l'éventail des membres

x_t = torch.from_numpy(x_grid).unsqueeze(1)
means_g, vars_g = ensemble1d.predict(x_t)   # [B, 300, 1]
means_g_np = means_g.squeeze(-1).detach().numpy()  # [B, 300]
vars_g_np  = vars_g.squeeze(-1).detach().numpy()   # [B, 300]

colors = ["steelblue", "darkorange", "forestgreen", "crimson", "purple"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_train, y_train, s=8, alpha=0.3, color="gray", label="Données", zorder=1)
ax.plot(x_grid, f_true(x_grid), "k--", lw=2, label="f vraie", zorder=2)
for b in range(ensemble1d.n_members):
    std_b = np.sqrt(vars_g_np[b])
    ax.plot(x_grid, means_g_np[b], color=colors[b], lw=1.5, alpha=0.8,
            label=f"Membre {b+1}")
    ax.fill_between(x_grid, means_g_np[b]-std_b, means_g_np[b]+std_b,
                    alpha=0.07, color=colors[b])
ax.axvspan(-1, 1, alpha=0.10, color="orange")
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
ax.set_title("Éventail des membres : consensus sur les données, divergence dans le trou")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

**Lecture.** L'image est sans ambiguïté : les $B$ courbes se **superposent** là où il y a des données
(les experts s'accordent → épistémique faible) et **s'éventent** dans le trou (les experts se
contredisent → épistémique forte). Chaque membre porte en plus sa propre bande aléatoire. On a donc
maintenant **les deux incertitudes au même endroit** : l'aléatoire dans l'épaisseur de chaque bande,
l'épistémique dans l'écartement des moyennes. Il ne reste qu'à les séparer proprement.

## III.3 — Séparer aléatoire et épistémique, terme par terme

Avec $B$ membres prédisant chacun une distribution gaussienne,

$$p_b(y \mid x) = \mathcal{N}\!\big(\mu_b(x), \sigma_b^2(x)\big),$$

l'ensemble ne produit pas une seule gaussienne exacte, mais un **mélange** de gaussiennes : on choisit
un membre $b$, puis on échantillonne dans la gaussienne prédite par ce membre.

La moyenne du mélange est simplement la moyenne des moyennes :

$$\mu_\star(x) = \frac{1}{B}\sum_{b=1}^{B}\mu_b(x)$$

La variance totale se décompose en deux morceaux :

$$\operatorname{Var}_\star(x) =
\underbrace{\frac{1}{B}\sum_{b=1}^{B}\sigma_b^2(x)}_{\textbf{aléatoire}}
\;+\;
\underbrace{\frac{1}{B}\sum_{b=1}^{B}\mu_b^2(x) - \mu_\star^2(x)}_{\textbf{épistémique}}$$

Le premier terme regarde **à l'intérieur** de chaque expert : même si l'on faisait confiance à un
membre donné, quelle variance annonce-t-il autour de sa propre moyenne ? C'est l'incertitude
aléatoire.

Le second terme regarde **entre** les experts : les moyennes $\mu_b(x)$ sont-elles proches ou
dispersées ? On reconnaît la formule classique de la variance :

$$\operatorname{Var}_b(\mu_b) = \mathbb{E}_b[\mu_b^2] - \mathbb{E}_b[\mu_b]^2$$

| Terme | Lecture |
|-------|---------|
| $\mu_\star$ | prédiction consensus du comité |
| $\frac1B\sum_b \sigma_b^2$ | **aléatoire** : moyenne des bruits déclarés par les membres |
| $\frac1B\sum_b \mu_b^2 - \mu_\star^2$ | **épistémique** : dispersion des prédictions moyennes |
| $\operatorname{Var}_\star$ | incertitude totale utilisée pour comprendre la fiabilité du modèle |

Donc PETS sépare deux questions différentes :

- *si je choisis un expert, à quel point dit-il que la transition est bruitée ?*
- *si je change d'expert plausible, est-ce que la prédiction change beaucoup ?*

C'est exactement la loi des variances totales de la section I.2, rendue calculable par l'ensemble.
Le second terme n'est rien d'autre que la **variance empirique des $\mu_b$** : le désaccord, chiffré.

In [ ]:
# Décomposition aléatoire / épistémique sur la grille

# means_g_np : [B, 300]   vars_g_np : [B, 300]
mu_star    = means_g_np.mean(axis=0)                # [300]
aleatoric  = vars_g_np.mean(axis=0)                 # [300]  — moyenne des variances déclarées
epistemic  = means_g_np.var(axis=0)                 # [300]  — variance des moyennes (désaccord)
total      = aleatoric + epistemic

# Vérification de l'identité : la variance du MÉLANGE de gaussiennes vaut
#   Var_totale = E[sigma_b^2 + mu_b^2] - mu_star^2,
# et on retrouve bien aléatoire + épistémique (loi des variances totales).
total_mixture = (vars_g_np + means_g_np ** 2).mean(axis=0) - mu_star ** 2
assert np.allclose(total_mixture, total, atol=1e-5), "Décomposition incohérente avec la variance du mélange"

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_grid, aleatoric,  color="steelblue",   lw=2, label="Aléatoire  $\\frac{1}{B}\\sum\\sigma_b^2$")
ax.plot(x_grid, epistemic,  color="darkorange",  lw=2, label="Épistémique  $\\mathrm{Var}(\\mu_b)$")
ax.plot(x_grid, total,      color="gray",        lw=1.5, ls="--", label="Totale")
ax.axvspan(-1, 1, alpha=0.10, color="orange", label="Trou")
ax.set_title("Décomposition de la variance : aléatoire vs épistémique")
ax.set_xlabel("x"); ax.set_ylabel("Variance")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Lecture.** Les deux courbes racontent des histoires opposées. L'**aléatoire** (bleu) suit le bruit
des données : faible à gauche, élevé à droite, et reste *modéré dans le trou* (le réseau n'invente pas
de bruit là où il n'a rien vu). L'**épistémique** (orange) fait l'inverse : quasi nulle sur les
données, elle **explose dans le trou**. C'est la signature qu'on cherchait : « ça, c'est du bruit
qu'on ne réduira pas » vs « ça, c'est de l'ignorance qu'une visite réglerait ». PETS dispose enfin
d'un modèle qui connaît ses limites.

On a tout testé sur un jouet 1D pour *voir* les idées. Passons à un vrai système à apprendre :
HalfCheetah, 18 dimensions d'état, 6 d'action. Le même ensemble probabiliste va y modéliser la
dynamique — avec la même décomposition d'incertitude, invisible à l'œil mais essentielle au planning.

---
# Partie IV — Apprendre la dynamique de HalfCheetah

## IV.1 — HalfCheetah-v5, et pourquoi on augmente l'observation

HalfCheetah est un guépard 2D (physique MuJoCo) qu'on veut faire **courir vers l'avant** le plus vite
possible. L'observation standard contient les angles et vitesses des articulations, mais **pas la
position horizontale $x$** du torse (elle est masquée par défaut). Or la récompense de HalfCheetah est
$r = \frac{x_{t+1}-x_t}{\Delta t} - 0.1\,\lVert a\rVert^2$ : elle **dépend de $x$**.

Pour pouvoir **calculer la récompense à partir des états prédits** par le modèle — indispensable au
planning — on demande à l'environnement d'**inclure $x$** dans l'observation
(`exclude_current_positions_from_observation=False`). L'observation passe ainsi à **18 dimensions**,
avec $x$ en **index 0**.

| Spec | Valeur |
|------|--------|
| Observation | 18-dim (position $x$ en index 0, puis angles/vitesses) |
| Action | 6-dim continue, bornée $[-1, 1]$ (couples articulaires) |
| Récompense | $\frac{x_{t+1}-x_t}{\Delta t} - 0.1\lVert a\rVert^2$ (avance − coût de contrôle) |
| $\Delta t$ | $0.05$ s (`env.unwrapped.dt`) |

In [ ]:
# Créer l'environnement HalfCheetah avec x inclus dans l'observation

env = gym.make("HalfCheetah-v5", exclude_current_positions_from_observation=False)
obs, _ = env.reset(seed=0)
print(f"obs.shape     = {obs.shape}   (attendu: 18)")
print(f"action_space  = {env.action_space}")
print(f"action low/high: {env.action_space.low[:3]} ... / {env.action_space.high[:3]} ...")
print(f"dt            = {env.unwrapped.dt}")

# Épisode aléatoire court pour voir que l'agent aléatoire ne va nulle part
np.random.seed(0)
obs, _ = env.reset(seed=0)
x_positions = [float(obs[0])]
for _ in range(50):
    a = env.action_space.sample()
    obs, r, terminated, truncated, _ = env.step(a)
    x_positions.append(float(obs[0]))
    if terminated or truncated:
        obs, _ = env.reset()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_positions, color="steelblue", lw=2, marker="o", markersize=3)
ax.set_xlabel("Pas de temps")
ax.set_ylabel("Position x du torse")
ax.set_title("Agent aléatoire sur HalfCheetah : position x (piétinement)")
plt.tight_layout()
plt.show()

## IV.2 — Prédire la variation $\Delta s$, et tout normaliser

Comme pour PILCO, on n'apprend pas directement $s \mapsto s'$, mais plutôt :

$$\Delta s = s' - s$$

puis on reconstruit ensuite :

$$s' = s + \Delta s$$

Pourquoi ce détour ? Parce qu'à l'échelle d'un pas de simulation, l'état change souvent **moins** que
sa valeur absolue. La position du torso, par exemple, peut dériver au fil de l'épisode ; mais son
incrément entre deux pas reste beaucoup plus régulier. Le réseau apprend donc une correction locale :
*étant donné où je suis et quelle action j'applique, de combien l'état bouge-t-il ?*

C'est généralement plus facile que de redire tout l'état suivant depuis zéro. On peut le voir comme un
modèle résiduel : le modèle n'a pas besoin de réapprendre l'identité $s' \approx s$, seulement la
partie dynamique $\Delta s$.

Second point crucial pour des **réseaux** : la **normalisation**. Les dimensions de HalfCheetah n'ont
pas du tout les mêmes échelles. Une coordonnée de position, une vitesse angulaire, une vitesse
linéaire et un couple moteur ne vivent pas dans les mêmes ordres de grandeur. Si on donne tout tel
quel au réseau, les grandes dimensions dominent la loss et les gradients, tandis que les petites
peuvent devenir presque invisibles.

On standardise donc :

- les entrées $[s,a]$ ;
- les cibles $\Delta s$.

Concrètement, pour une variable $z$ du dataset :

$$z_{\text{norm}} = \frac{z - \mu_z}{\sigma_z + \varepsilon}$$

Le réseau travaille dans cet espace normalisé, puis on dé-normalise ses sorties pour revenir dans les
unités de l'environnement.

Cette étape n'est pas un détail cosmétique : elle change directement la qualité de l'optimisation.
Sans normalisation, le réseau passe une partie de sa capacité à corriger des problèmes d'échelle au
lieu d'apprendre la dynamique. La NLL devient aussi plus difficile à interpréter, parce que la variance
prédite doit compenser des dimensions aux amplitudes très différentes.

Dans PILCO avec GP, une partie de ce rôle était portée par les *length-scales* ARD du kernel : chaque
dimension pouvait avoir sa propre échelle de pertinence. Avec un réseau de neurones, on fait cette
mise à l'échelle explicitement dans le preprocessing.

In [ ]:
# Collecte de transitions aléatoires + entraînement de l'ensemble dynamique

def collect_transitions(env, n_steps, seed=0):
    # Retourne X=[s,a] (24-dim), Y=delta_s (18-dim), en float32
    obs_list, act_list, next_obs_list = [], [], []
    obs, _ = env.reset(seed=seed)
    for _ in range(n_steps):
        action = env.action_space.sample()
        next_obs, _, terminated, truncated, _ = env.step(action)
        obs_list.append(obs.copy())
        act_list.append(action.copy())
        next_obs_list.append(next_obs.copy())
        if terminated or truncated:
            obs, _ = env.reset()
        else:
            obs = next_obs
    X = np.concatenate([np.array(obs_list), np.array(act_list)], axis=1).astype(np.float32)
    Y = (np.array(next_obs_list) - np.array(obs_list)).astype(np.float32)
    return X, Y

print("Collecte de 4000 transitions aléatoires...")
X_hc, Y_hc = collect_transitions(env, n_steps=4000, seed=42)
print(f"X: {X_hc.shape}  Y: {Y_hc.shape}")

# Split train/held-out (90/10)
N_hc = len(X_hc)
n_train_hc = int(0.9 * N_hc)
perm = np.random.permutation(N_hc)
X_tr_hc = torch.from_numpy(X_hc[perm[:n_train_hc]])
Y_tr_hc = torch.from_numpy(Y_hc[perm[:n_train_hc]])
X_ho_hc = torch.from_numpy(X_hc[perm[n_train_hc:]])
Y_ho_hc = torch.from_numpy(Y_hc[perm[n_train_hc:]])

OBS_DIM = 18
ACT_DIM = 6

# ============================================================
# Ensemble HalfCheetah avec normalisation intégrée
# ============================================================


In [ ]:
class NormalizedEnsemble(nn.Module):
    # Ensemble probabiliste avec normalisation entrée/sortie intégrée
    # (buffers non entraînables stockés dans l'état du module)

    def __init__(self, input_dim, output_dim, n_members=5, hidden_dim=128, n_layers=3):
        super().__init__()
        self.input_dim  = input_dim
        self.output_dim = output_dim
        self.n_members  = n_members
        self.members = nn.ModuleList([
            ProbabilisticMLP(input_dim, output_dim, hidden_dim=hidden_dim, n_layers=n_layers)
            for _ in range(n_members)
        ])
        # Buffers de normalisation (float32)
        self.register_buffer("x_mean", torch.zeros(input_dim))
        self.register_buffer("x_std",  torch.ones(input_dim))
        self.register_buffer("y_mean", torch.zeros(output_dim))
        self.register_buffer("y_std",  torch.ones(output_dim))

    def fit_normalizer(self, X, Y):
        self.x_mean.copy_(X.mean(0))
        self.x_std.copy_(X.std(0).clamp_min(1e-6))
        self.y_mean.copy_(Y.mean(0))
        self.y_std.copy_(Y.std(0).clamp_min(1e-6))

    def _norm_x(self, X): return (X - self.x_mean) / self.x_std
    def _norm_y(self, Y): return (Y - self.y_mean) / self.y_std
    def _denorm_y(self, Y_n): return Y_n * self.y_std + self.y_mean

    def fit(self, X, Y, steps=300, lr=1e-3, batch_size=256):
        self.fit_normalizer(X, Y)
        Xn = self._norm_x(X)
        Yn = self._norm_y(Y)
        N  = Xn.shape[0]
        nll_history = []
        for b, member in enumerate(self.members):
            idx = torch.randint(0, N, (N,))
            Xb, Yb = Xn[idx], Yn[idx]
            opt = torch.optim.Adam(member.parameters(), lr=lr)
            member.train()
            last_nll = 0.0
            for _ in range(steps):
                bi = torch.randint(0, N, (min(batch_size, N),))
                mu, lv = member(Xb[bi])
                loss = gaussian_nll(mu, lv, Yb[bi]) + 0.01*(member.max_logvar.sum()-member.min_logvar.sum())
                opt.zero_grad(); loss.backward(); opt.step()
                last_nll = float(loss.item())
            member.eval()
            nll_history.append(last_nll)
        return nll_history

    @torch.no_grad()
    def predict(self, x):
        # x : raw (non-normalisé), shape [N, input_dim]
        # retourne means [B,N,out], vars [B,N,out] en espace physique
        xn = self._norm_x(x)
        all_means, all_vars = [], []
        for m in self.members:
            mu_n, lv = m(xn)
            mu = mu_n * self.y_std + self.y_mean
            var = torch.exp(lv) * (self.y_std ** 2)
            all_means.append(mu.unsqueeze(0))
            all_vars.append(var.unsqueeze(0))
        return torch.cat(all_means, 0), torch.cat(all_vars, 0)

    @torch.no_grad()
    def propagate(self, states, actions, model_idx):
        # Propagation d'un pas pour CEM (vectorisée)
        # states [B, obs], actions [B, act], model_idx [B] -> next_states [B, obs]
        B = states.shape[0]
        x = torch.cat([states, actions], dim=-1)
        xn = self._norm_x(x)
        means_n  = torch.zeros(B, self.output_dim)
        logvars_n = torch.zeros(B, self.output_dim)
        for m_i, member in enumerate(self.members):
            mask = (model_idx == m_i)
            if not mask.any(): continue
            mn, lv = member(xn[mask])
            means_n[mask]   = mn
            logvars_n[mask] = lv
        eps   = torch.randn_like(means_n)
        std_n = torch.exp(0.5 * logvars_n)
        delta_n = means_n + std_n * eps
        delta   = delta_n * self.y_std + self.y_mean
        return states + delta

    def disagreement(self, X):
        means, _ = self.predict(X)       # [B, N, D]
        return float(means.std(dim=0).mean().item())


torch.manual_seed(0)
np.random.seed(0)
dyn = NormalizedEnsemble(OBS_DIM+ACT_DIM, OBS_DIM, n_members=5, hidden_dim=128, n_layers=3)
print(f"Entraînement de l'ensemble ({dyn.n_members} membres, hidden=128, 300 steps Adam)…")
nll_hist = dyn.fit(X_tr_hc, Y_tr_hc, steps=300, lr=1e-3, batch_size=256)
print(f"NLL finale par membre : {[f'{v:.3f}' for v in nll_hist]}")


In [ ]:
# RMSE held-out + désaccord in-distribution vs hors-distribution

with torch.no_grad():
    means_ho, _ = dyn.predict(X_ho_hc)        # [B, N_ho, 18]
    mu_star_ho  = means_ho.mean(0)            # [N_ho, 18]

    # Dé-normaliser Y_ho pour comparer en espace physique
    delta_pred = mu_star_ho                    # déjà dénormalisé par predict()
    delta_true = Y_ho_hc
    rmse_per_dim = ((delta_pred - delta_true)**2).mean(0).sqrt()  # [18]
    rmse_global  = float(rmse_per_dim.mean().item())

print(f"RMSE held-out global : {rmse_global:.5f}  (en unités physiques Δs)")

# Désaccord in-distribution
disc_in  = dyn.disagreement(X_ho_hc)

# Désaccord hors-distribution : états bruités (loin du support)
X_ood = X_ho_hc + torch.randn_like(X_ho_hc) * 5.0
disc_ood = dyn.disagreement(X_ood)

print(f"Désaccord in-distribution  : {disc_in:.5f}")
print(f"Désaccord hors-distribution: {disc_ood:.5f}  (doit être > in-dist)")
assert disc_ood > disc_in, "Le désaccord devrait monter hors-distribution"

# Barplot RMSE par dimension
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(range(OBS_DIM), rmse_per_dim.numpy(), color="steelblue", alpha=0.8)
ax.set_xlabel("Dimension de l'état"); ax.set_ylabel("RMSE (physique)")
ax.set_title(f"RMSE Δs par dimension (global={rmse_global:.4f})")
plt.tight_layout()
plt.show()

**Lecture.** Avec quelques milliers de transitions aléatoires, l'ensemble prédit déjà $\Delta s$ avec
une erreur faible sur des états *vus*, et son **désaccord grimpe sur les états hors-distribution** —
notre boussole épistémique fonctionne en 18 dimensions comme sur le jouet 1D. Mais prédire **un pas**
ne suffit pas : pour planifier, il faut **enchaîner** des dizaines de pas dans le modèle, en
transportant l'incertitude. C'est la propagation par particules (Partie V).

---
# Partie V — Trajectory sampling : propager des particules

## V.1 — Pourquoi des particules

Dans les méthodes model-based, le problème n'est pas seulement de prédire **le prochain état**. Le
vrai problème est de prédire ce qui arrive quand on répète cette opération plusieurs fois :

$$s_t \rightarrow s_{t+1} \rightarrow s_{t+2} \rightarrow \cdots \rightarrow s_{t+H}$$

Or chaque petite erreur de modèle peut se propager. Une prédiction légèrement trop optimiste à
$t+1$ peut placer le modèle dans une région différente à $t+2$, puis encore plus différente à $t+3$.
Donc, pour planifier correctement, on doit propager non seulement une trajectoire moyenne, mais aussi
**l'incertitude sur cette trajectoire**.

PILCO le faisait analytiquement. Il décrivait la croyance sur l'état par une gaussienne :

$$s_t \sim \mathcal{N}(\mu_t, \Sigma_t)$$

puis utilisait le GP pour approximer les moments de l'état suivant :

$$s_{t+1} \approx \mathcal{N}(\mu_{t+1}, \Sigma_{t+1})$$

Autrement dit, PILCO résumait toute la croyance par deux objets : une moyenne et une covariance. C'est
élégant, mais cela repose sur une structure mathématique spéciale : un GP avec un kernel pour lequel
on peut calculer certains moments en forme fermée.

PETS change de représentation. Son modèle de dynamique est un **ensemble de réseaux probabilistes**.
Chaque membre prédit une gaussienne sur la variation d'état :

$$f_b(s,a) = \mathcal{N}\!\big(\mu_b(s,a), \sigma_b^2(s,a)\big)$$

L'ensemble complet forme donc un **mélange** de gaussiennes. Et après plusieurs pas dans des réseaux
non linéaires, ce mélange peut devenir très compliqué : asymétrique, étiré, multimodal, ou dépendant
fortement du membre d'ensemble choisi. Il n'existe plus de petite formule propre du type
$\mu_{t+1}, \Sigma_{t+1}$.

La solution de PETS est pragmatique : **représenter la distribution par un nuage d'échantillons**.

On maintient $P$ particules :

$$\{s_t^{(1)}, s_t^{(2)}, \dots, s_t^{(P)}\}$$

Une particule est simplement **un état possible du monde**. Ce n'est pas un agent, ce n'est pas un
réseau, ce n'est pas une copie de l'environnement réel. C'est une hypothèse : *si les incertitudes se
réalisent de cette façon, alors le système pourrait être dans cet état*.

On peut lire le nuage de particules comme une distribution empirique :

$$p(s_t) \approx \frac{1}{P}\sum_{p=1}^{P}\delta\!\left(s_t - s_t^{(p)}\right)$$

Chaque particule est une petite masse de probabilité placée sur un état concret. Si beaucoup de
particules sont regroupées, le modèle croit que cette région est probable. Si elles sont dispersées,
le modèle reconnaît que plusieurs futurs sont plausibles.

Cette interprétation est importante :

- la **moyenne** des particules donne une trajectoire centrale ;
- la **variance** des particules donne l'incertitude propagée ;
- les particules éloignées montrent les scénarios moins probables mais possibles ;
- une distribution multimodale peut être représentée naturellement par plusieurs groupes de
  particules, alors qu'une seule gaussienne la résumerait mal.

C'est pour cela que les particules sont plus flexibles qu'un simple couple $(\mu, \Sigma)$. Une
gaussienne dit essentiellement : *je suis autour de cette moyenne, avec cette ellipse d'incertitude*.
Un nuage de particules peut dire : *il y a deux futurs plausibles très différents*, ou *la distribution
est courbée*, ou *quelques scénarios partent très loin*.

Une bonne analogie est la météo. Au lieu de prédire une seule carte pour demain, les météorologues
lancent plusieurs simulations avec de petites variations initiales et différents modèles. Chaque
simulation est une particule au niveau de la trajectoire. Si toutes les simulations restent proches,
la prévision est fiable. Si elles se séparent rapidement, l'avenir est incertain.

PETS fait la même chose pour le robot. Pour une séquence d'actions candidate, il ne demande pas :

> quelle est la trajectoire moyenne prédite ?

Il demande plutôt :

> si je simule beaucoup de futurs plausibles sous cette séquence d'actions, quelle récompense moyenne
> obtiens-je, et à quel point ces futurs se dispersent-ils ?

Concrètement, pour évaluer une séquence d'actions, PETS duplique l'état courant en $P$ particules,
puis les fait avancer dans le modèle pendant $H$ pas. À chaque pas, une particule peut évoluer
différemment parce qu'elle subit :

- l'incertitude **épistémique** : elle peut être propagée par un membre différent de l'ensemble ;
- l'incertitude **aléatoire** : le membre choisi échantillonne une transition autour de sa moyenne.

Après $H$ pas, on obtient non pas une trajectoire, mais un **faisceau de trajectoires**. Le score de
la séquence d'actions est estimé en moyennant les récompenses sur ce faisceau :

$$\hat{J}(a_{t:t+H-1}) =
\frac{1}{P}\sum_{p=1}^{P}\sum_{h=0}^{H-1}
r\!\left(s_{t+h}^{(p)}, a_{t+h}\right)$$

Cette moyenne protège contre les plans trop fragiles. Si une séquence d'actions ne marche que dans un
futur chanceux mais échoue dans beaucoup d'autres particules, son score moyen baisse. À l'inverse, une
séquence qui fonctionne sur la majorité des futurs plausibles devient attractive pour le planificateur.

Le prix à payer est computationnel. Si l'on teste $N$ séquences d'actions, avec $P$ particules, sur un
horizon $H$, on fait environ :

$$N \times P \times H$$

appels au modèle. PETS accepte ce coût, car les réseaux et les particules se vectorisent bien sur GPU,
et cette approximation reste beaucoup plus scalable que les calculs GP classiques de PILCO quand le
dataset grandit.

## V.2 — Un pas de propagation, terme par terme

Prenons une particule $p$ à l'état $s_t^{(p)}$, une action $a_t$, et un membre d'ensemble assigné
$b(p)$.

Le membre choisi prédit une distribution sur la variation d'état :

$$\big(\mu_t^{(p)}, \sigma_t^{2,(p)}\big)
= f_{b(p)}\big(s_t^{(p)}, a_t\big)$$

La moyenne $\mu_t^{(p)}$ dit : *si ce membre a raison, voilà le déplacement attendu*. La variance
$\sigma_t^{2,(p)}$ dit : *même pour ce membre, cette transition reste plus ou moins bruitée*.

Pour obtenir un futur concret, on échantillonne une variation d'état :

$$\Delta s_t^{(p)}
= \mu_t^{(p)} + \sigma_t^{(p)} \odot \varepsilon_t^{(p)},
\qquad
\varepsilon_t^{(p)} \sim \mathcal{N}(0, I)$$

puis on avance la particule :

$$s_{t+1}^{(p)} = s_t^{(p)} + \Delta s_t^{(p)}$$

| Terme | Rôle |
|-------|------|
| $s_t^{(p)}$ | l'état actuel de la particule $p$ |
| $a_t$ | l'action testée à ce pas de planification |
| $b(p)$ | le membre de l'ensemble utilisé pour cette particule |
| $f_{b(p)}$ | le réseau probabiliste qui prédit $\mu$ et $\sigma^2$ |
| $\mu_t^{(p)}$ | la variation moyenne prédite par ce membre |
| $\sigma_t^{(p)} \odot \varepsilon_t^{(p)}$ | un bruit gaussien local, dimension par dimension |
| $\Delta s_t^{(p)}$ | une variation d'état plausible |
| $s_t^{(p)} + \Delta s_t^{(p)}$ | l'état suivant reconstruit par modèle résiduel |

Cette petite équation contient toute la philosophie PETS.

D'abord, la particule porte l'**incertitude sur l'état futur**. Après un pas, elle représente un état
possible. Après plusieurs pas, elle devient une trajectoire possible.

Ensuite, le choix du membre $b(p)$ porte l'**incertitude épistémique**. Si les membres de l'ensemble
ne sont pas d'accord, deux particules identiques mais assignées à deux membres différents peuvent
partir dans des directions différentes. C'est exactement ce qu'on veut : dans les zones peu connues,
le modèle doit produire des futurs divergents.

Enfin, le bruit $\varepsilon_t^{(p)}$ porte l'**incertitude aléatoire**. Même si l'on gardait le même
membre, celui-ci peut annoncer que la transition est intrinsèquement bruitée. On échantillonne donc
autour de sa moyenne.

La valeur d'une séquence d'actions est ensuite estimée en moyennant les récompenses sur toutes les
particules :

$$\hat{J}(a_{t:t+H-1}) =
\frac{1}{P}\sum_{p=1}^{P}\sum_{h=0}^{H-1} r\!\left(s_{t+h}^{(p)}, a_{t+h}\right)$$

Si une séquence marche seulement dans quelques futurs chanceux mais échoue dans beaucoup d'autres,
sa moyenne sera mauvaise. Si elle donne de bons retours sur la majorité des particules, elle est plus
robuste. C'est pour cela que PETS ne planifie pas dans une trajectoire unique : il planifie dans un
**nuage de futurs plausibles**.

In [ ]:
# Propagation d'un pas pour un nuage de particules (vectorisée par membre)

def propagate_step(ensemble, states, actions, model_idx):
    # states    : [P, obs_dim]
    # actions   : [P, act_dim]
    # model_idx : [P] — indice du membre assigné à chaque particule
    # retourne  : next_states [P, obs_dim]
    return ensemble.propagate(states, actions, model_idx)

# Mini-test : shapes + reproductibilité à seed fixe
torch.manual_seed(7)
P_test = 12
s0 = torch.from_numpy(env.reset(seed=0)[0]).unsqueeze(0).expand(P_test, -1).float().clone()
a0 = torch.zeros(P_test, ACT_DIM)
idx0 = torch.arange(P_test) % dyn.n_members

torch.manual_seed(7)
ns1 = propagate_step(dyn, s0, a0, idx0)
torch.manual_seed(7)
ns2 = propagate_step(dyn, s0, a0, idx0)

assert ns1.shape == (P_test, OBS_DIM), f"shape: {ns1.shape}"
assert torch.allclose(ns1, ns2), "Résultat non reproductible à seed fixe"

print(f"✓ propagate_step : shape {ns1.shape}, reproductible à seed fixe")

## V.3 — TS1 vs TS∞ : changer d'expert, ou pas

Il reste une décision importante : **quel membre de l'ensemble fait avancer chaque particule pendant
l'horizon de planification ?**

Rappelons le rôle d'une particule : elle représente un futur plausible. Mais ce futur plausible dépend
de deux choses :

1. le bruit aléatoire échantillonné à chaque transition ;
2. l'hypothèse de dynamique utilisée, c'est-à-dire le membre de l'ensemble.

C'est le deuxième point qui distingue TS1 et $\text{TS}\infty$.

### TS1 : rééchantillonner le modèle à chaque pas

En **TS1**, à chaque pas de temps, chaque particule retire un membre de l'ensemble :

$$b_t^{(p)} \sim \{1,\dots,B\}$$

Donc une même particule peut être propagée par le membre 3 au premier pas, puis le membre 1 au second,
puis le membre 5 au troisième.

Intuitivement, c'est comme changer d'expert chaque jour :

> lundi j'écoute l'expert 3, mardi l'expert 1, mercredi l'expert 5.

Cette stratégie mélange beaucoup les hypothèses. Elle peut donner une estimation raisonnable de la
moyenne, mais elle casse une propriété essentielle : une erreur de modèle épistémique devrait être
**corrélée dans le temps**.

Si un modèle sous-estime systématiquement une vitesse, il va probablement la sous-estimer pendant
plusieurs pas, pas seulement pendant une seule transition. En TS1, cette erreur peut être annulée au
pas suivant par un autre membre qui surestime. Les erreurs se compensent artificiellement.

Résultat : TS1 a tendance à **moyenner l'incertitude épistémique pas à pas** et peut sous-estimer le
risque des longs rollouts.

###  $\text{TS}\infty$ : garder le même modèle pendant toute la trajectoire

En **$\text{TS}\infty$**, chaque particule reçoit un membre une fois pour toutes :

$$b^{(p)} \sim \{1,\dots,B\}$$

puis ce même membre est utilisé pendant tout l'horizon :

$$s_t^{(p)} \xrightarrow{f_{b^{(p)}}} s_{t+1}^{(p)}
\xrightarrow{f_{b^{(p)}}} s_{t+2}^{(p)}
\xrightarrow{f_{b^{(p)}}} \cdots
\xrightarrow{f_{b^{(p)}}} s_{t+H}^{(p)}$$

Intuitivement, c'est comme garder le même expert pendant dix jours. Si l'expert 3 pense que le sol est
glissant, il le pense du début à la fin. La trajectoire devient alors cohérente avec **une hypothèse
complète** sur la dynamique.

C'est exactement ce qu'on veut pour représenter l'incertitude épistémique. Les membres de l'ensemble
ne sont pas seulement des générateurs de bruit indépendant : ce sont des hypothèses globales sur le
monde. Une particule $\text{TS}\infty$ répond à la question :

> que se passerait-il si cette hypothèse de dynamique était la bonne ?

Puis l'ensemble des particules répond :

> que se passerait-il sous plusieurs hypothèses plausibles ?

La dispersion entre particules de membres différents devient donc une incertitude épistémique
**propagée et corrélée dans le temps**. Elle ne disparaît pas artificiellement à chaque pas.

| Stratégie | Membre utilisé | Interprétation | Effet sur l'épistémique |
|---|---|---|---|
| TS1 | nouveau membre à chaque pas | mélange d'experts au fil de la trajectoire | peut moyenner les erreurs et sous-estimer le risque |
| $\text{TS}\infty$ | même membre pour tout l'horizon | trajectoire cohérente sous une hypothèse de dynamique | conserve les erreurs corrélées dans le temps |

Le papier PETS recommande **$\text{TS}\infty$** pour cette raison, et c'est notre choix par défaut. C'est le même
principe que dans Deep PILCO : on gardait un masque de dropout fixe par particule pour que chaque
particule suive un modèle cohérent sur tout l'horizon. Ici, on garde un membre d'ensemble fixe pour
obtenir cette cohérence temporelle.

In [ ]:
# Rollout TS1 vs TS∞ depuis un état réel, séquence d'actions fixe

torch.manual_seed(42)
np.random.seed(42)

H_demo = 20
P_demo = 20   # particules
obs0, _ = env.reset(seed=5)
s_init  = torch.from_numpy(obs0).float()

# Séquence d'actions fixe (aléatoire mais figée)
torch.manual_seed(0)
fixed_actions = torch.randn(H_demo, ACT_DIM).clamp(-1, 1)  # [H, act_dim]

def rollout_particles(ensemble, s0, actions, P, mode="tsinf", seed=0):
    # Retourne trajectoires [H+1, P, obs_dim]
    torch.manual_seed(seed)
    states = s0.unsqueeze(0).expand(P, -1).clone()  # [P, obs_dim]
    if mode == "tsinf":
        model_idx = torch.arange(P) % ensemble.n_members  # fixe pour tout le rollout
    traj = [states.clone()]
    for t in range(actions.shape[0]):
        act_t = actions[t].unsqueeze(0).expand(P, -1)  # [P, act_dim]
        if mode == "ts1":
            model_idx = torch.randint(0, ensemble.n_members, (P,))
        next_states = propagate_step(ensemble, states, act_t, model_idx)
        states = next_states
        traj.append(states.clone())
    return torch.stack(traj, dim=0)  # [H+1, P, obs_dim]

traj_tsinf = rollout_particles(dyn, s_init, fixed_actions, P_demo, mode="tsinf", seed=0)
traj_ts1   = rollout_particles(dyn, s_init, fixed_actions, P_demo, mode="ts1",   seed=0)
# shape : [H+1, P, 18]

print(f"Trajectoires calculées — shape : {traj_tsinf.shape}")
print(f"(H+1={H_demo+1} pas, P={P_demo} particules, obs={OBS_DIM} dim)")

In [ ]:
# Visualisation : étalement TS1 vs TS∞ (dimension 8 = x_velocity)

dim_show = 8   # vitesse avant

tsinf_np = traj_tsinf[:, :, dim_show].detach().numpy()  # [H+1, P]
ts1_np   = traj_ts1[:,   :, dim_show].detach().numpy()

t_axis = np.arange(H_demo + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for mode_data, mode_name, ax in zip(
        [tsinf_np, ts1_np], ["TS∞ (membre fixe)", "TS1 (membre aléatoire)"], axes):
    mu_m  = mode_data.mean(axis=1)
    std_m = mode_data.std(axis=1)
    for p in range(P_demo):
        ax.plot(t_axis, mode_data[:, p], color="steelblue", alpha=0.25, lw=0.8)
    ax.plot(t_axis, mu_m,  color="navy", lw=2, label="Moyenne")
    ax.fill_between(t_axis, mu_m-std_m, mu_m+std_m,
                    alpha=0.3, color="steelblue", label=r"$\pm\sigma$")
    ax.set_title(mode_name)
    ax.set_xlabel("Pas de temps")
    ax.legend(fontsize=9)
axes[0].set_ylabel(f"Dim {dim_show} (x_velocity)")
fig.suptitle("Étalement du nuage de particules : TS∞ vs TS1", fontsize=12)
plt.tight_layout()
plt.show()

**Lecture.** En **$\text{TS}\infty$**, les particules tracent des trajectoires lisses et nettement séparées : chaque
« expert » déroule sa propre hypothèse de bout en bout, et l'éventail reflète une incertitude
épistémique **honnête et corrélée**. En **TS1**, les trajectoires sont plus emmêlées et l'éventail
souvent **plus resserré** : le brassage des membres à chaque pas moyenne l'épistémique et tend à la
**sous-estimer**. Pour un planificateur, sous-estimer l'incertitude = surconfiance = plans fragiles.
On garde donc **$\text{TS}\infty$**.

In [ ]:
# Mini-test modes TS∞ et TS1

# TS∞ : deux rollouts avec le même seed doivent être identiques
torch.manual_seed(42)
tr_a = rollout_particles(dyn, s_init, fixed_actions, 8, mode="tsinf", seed=99)
torch.manual_seed(42)
tr_b = rollout_particles(dyn, s_init, fixed_actions, 8, mode="tsinf", seed=99)
assert torch.allclose(tr_a, tr_b), "TS∞ non reproductible à seed fixe"

# TS1 : vérifier que les membres changent entre pas (au moins une particule)
# On teste en faisant un rollout court et en inspectant la variabilité
torch.manual_seed(0)
states_ts1 = s_init.unsqueeze(0).expand(10, -1).clone()
idx_step0 = torch.randint(0, dyn.n_members, (10,))
idx_step1 = torch.randint(0, dyn.n_members, (10,))
# Avec seed aléatoire, les deux tirages diffèrent (au moins sur quelques positions)
assert not torch.all(idx_step0 == idx_step1), "TS1 : indices identiques aux deux pas — anormal"

print("✓ TS∞ reproductible ; TS1 change bien d'indices entre pas")

Les particules nous donnent enfin de quoi **noter** n'importe quelle séquence d'actions : on la
propage, on accumule la récompense, on moyenne sur les particules → un rendement attendu. Il ne reste
plus qu'à **chercher** la meilleure séquence. C'est le rôle de l'optimiseur CEM (Partie VI).

---
# Partie VI — Planning : MPC + CEM

## VI.1 — Planifier, plutôt qu'apprendre une politique

PILCO apprenait une **politique paramétrique** : une fonction $\pi_\theta(s)$ optimisée hors-ligne
dans le modèle. Une fois entraînée, cette politique transforme directement un état en action.

PETS fait un choix presque opposé : il n'apprend pas de politique globale. À chaque pas réel, il
regarde l'état courant $s_t$ et résout un petit problème d'optimisation local :

$$a_{t:t+H-1}^\star
= \arg\max_{a_{t:t+H-1}}
\mathbb{E}\!\left[\sum_{h=0}^{H-1} r(s_{t+h}, a_{t+h})\right]$$

Autrement dit, PETS demande :

> parmi les séquences d'actions possibles à partir de maintenant, laquelle semble donner le meilleur
> futur dans mon modèle ?

Ce choix a deux avantages pédagogiquement importants.

D'abord, si le modèle est déjà raisonnable, le contrôle peut être bon **immédiatement** : on n'attend
pas qu'une politique apprenne par gradient sur des milliers d'épisodes. Ensuite, quand le modèle
s'améliore avec de nouvelles transitions, le planificateur en profite automatiquement. Il n'y a pas
de politique à réentraîner : le même algorithme de planning utilise juste un meilleur modèle.

Le prix à payer est clair : il faut planifier **à chaque pas**. PETS échange donc du calcul en ligne
contre de l'efficacité en données. C'est souvent un bon compromis en model-based RL : les transitions
réelles coûtent cher, mais les simulations dans le modèle sont relativement bon marché.

L'analogie est celle du joueur d'échecs. Il n'a pas une action mémorisée pour chaque position possible.
Il regarde la position actuelle, imagine plusieurs continuations, choisit le meilleur premier coup,
puis recommence à réfléchir après la réponse adverse. PETS fait pareil : il ne mémorise pas une
politique, il **réfléchit à chaque état**.

## VI.2 — MPC : l'horizon fuyant

Le mécanisme qui rend ce planning robuste s'appelle **MPC** : *Model Predictive Control*.

À l'état courant $s_t$ :

1. on cherche une séquence d'actions candidate sur un horizon $H$ :

   $$a_{t:t+H-1} = (a_t, a_{t+1}, \dots, a_{t+H-1})$$

2. on évalue cette séquence dans le modèle, avec des particules ;
3. on choisit la meilleure séquence trouvée ;
4. on **n'exécute que la première action** $a_t$ dans le vrai monde ;
5. on observe le vrai nouvel état $s_{t+1}$ ;
6. on recommence toute l'optimisation depuis ce nouvel état.

Ce détail est essentiel : PETS planifie $H$ pas, mais n'en joue qu'un.

Pourquoi jeter les $H-1$ actions restantes ? Parce que les prédictions deviennent moins fiables à
mesure que l'horizon avance. À un pas, le modèle est souvent correct. À dix ou vingt pas, les petites
erreurs s'accumulent. La vraie observation $s_{t+1}$ remet le système sur les rails : elle corrige
immédiatement la trajectoire imaginée.

C'est le sens de l'**horizon fuyant**. À chaque instant, l'horizon se décale d'un pas vers le futur.
On ne suit jamais aveuglément un vieux plan. On utilise le plan pour décider maintenant, puis on
replanifie avec la nouvelle information réelle.

On peut le résumer ainsi :

$$\text{planifier loin} \quad+\quad \text{agir court} \quad+\quad \text{corriger souvent}$$

C'est ce qui rend MPC beaucoup plus robuste qu'un plan en boucle ouverte exécuté jusqu'au bout.

### Petite visualisation : pourquoi replanifier ?

Le schéma suivant compare deux idées :

- **boucle ouverte** : on calcule un plan et on l'exécute entièrement ;
- **MPC** : on recalcule un plan après chaque observation réelle.





In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))

steps = np.arange(8)
true_path = np.array([0.0, 0.9, 1.7, 2.4, 3.0, 3.5, 3.9, 4.2])
open_loop_pred = np.array([0.0, 1.0, 2.0, 3.1, 4.4, 5.8, 7.3, 9.0])

# En MPC, chaque prédiction repart du vrai état observé.
mpc_segments = [
    (0, [0.0, 1.0, 2.0, 3.0]),
    (1, [0.9, 1.8, 2.7, 3.5]),
    (2, [1.7, 2.5, 3.2, 3.8]),
    (3, [2.4, 3.1, 3.7, 4.1]),
]

ax.plot(steps, true_path, "o-", lw=2.5, color="black", label="états réels observés")
ax.plot(steps, open_loop_pred, "--", lw=2, color="tab:red", label="plan initial exécuté en boucle ouverte")

for start, segment in mpc_segments:
    xs = np.arange(start, start + len(segment))
    ax.plot(xs, segment, color="tab:blue", alpha=0.45, lw=2)
    ax.scatter(xs[0], segment[0], color="tab:blue", s=35)

ax.plot([], [], color="tab:blue", alpha=0.6, lw=2, label="plans MPC recalculés")
ax.set_title("MPC : chaque plan est corrigé par la prochaine observation réelle")
ax.set_xlabel("temps")
ax.set_ylabel("position schématique")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Le plan rouge devient rapidement faux parce qu'il s'appuie sur ses propres prédictions.
Les segments bleus repartent régulièrement du vrai état observé : même si chaque plan est imparfait,
le contrôle reste ancré dans la réalité.

## VI.3 — CEM : la méthode d'entropie croisée

Il reste à résoudre le problème pratique : comment trouver une bonne séquence d'actions dans un espace
continu ?

Sur HalfCheetah, une action a plusieurs dimensions. Si l'horizon vaut $H$, une séquence complète vit
dans un espace de dimension :

$$H \times A$$

Même pour un horizon modeste, c'est un espace trop grand pour faire une grille. Et comme PETS évalue
les plans par échantillonnage de particules, on préfère un optimiseur simple, sans supposer que le
score est lisse ou facile à différencier.

C'est le rôle de **CEM** (*Cross-Entropy Method*).

CEM maintient une distribution sur les séquences d'actions. Dans notre cas, on prend une gaussienne
diagonale :

$$q(a_{0:H-1}) = \mathcal{N}(\mu, \sigma^2)$$

avec :

$$\mu \in \mathbb{R}^{H\times A},
\qquad
\sigma \in \mathbb{R}^{H\times A}$$

La moyenne $\mu$ représente le plan actuellement préféré. L'écart-type $\sigma$ représente l'ampleur
de l'exploration autour de ce plan.

À chaque itération CEM :

| Étape | Formule | Lecture |
|-------|---------|---------|
| **Échantillonner** | $a^{(n)} \sim \mathcal{N}(\mu, \sigma^2)$, puis clamp à $[-1,1]$ | tirer $N$ plans candidats |
| **Évaluer** | $R^{(n)} = \frac{1}{P}\sum_p\sum_h r(s_h^{(p)}, a_h^{(n)})$ | scorer chaque plan avec les particules |
| **Sélectionner** | garder les $K = \lceil \rho N\rceil$ meilleurs plans | ne conserver que les élites |
| **Réajuster** | $\mu \leftarrow (1-\alpha)\mu + \alpha\,\mu_{\text{élite}}$ | déplacer la distribution vers les bons plans |
| **Resserrer** | $\sigma \leftarrow (1-\alpha)\sigma + \alpha\,\sigma_{\text{élite}}$ | réduire l'exploration autour des élites |

L'intuition est celle d'un tireur qui ajuste sa visée. Au début, il tire large. Il regarde où sont les
meilleurs impacts, recentre sa visée, resserre la dispersion, puis recommence. Après quelques
itérations, la distribution se concentre sur une famille de plans prometteurs.

CEM n'a pas besoin de gradient. Il demande seulement : *donne-moi un score pour cette séquence
d'actions*. C'est pour cela qu'il se combine très bien avec PETS : le score vient d'un rollout
imaginé dans l'ensemble de modèles.

In [ ]:
### Petite visualisation : CEM se resserre sur les bons plans

rng = np.random.default_rng(3)

def toy_return(a):
    # Fonction jouet : maximum autour de a=0.7, avec un petit maximum local à gauche.
    return (
        1.3 * np.exp(-0.5 * ((a - 0.7) / 0.18) ** 2)
        + 0.45 * np.exp(-0.5 * ((a + 0.45) / 0.25) ** 2)
        - 0.05 * a**2
    )

xs = np.linspace(-1.4, 1.4, 400)
mu, sigma = -0.4, 0.8
elite_frac = 0.15
alpha = 0.8
population = 180

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)

for it, ax in enumerate(axes):
    samples = rng.normal(mu, sigma, size=population)
    samples = np.clip(samples, -1.2, 1.2)
    returns = toy_return(samples)
    elite_count = max(1, int(elite_frac * population))
    elite_idx = np.argsort(returns)[-elite_count:]
    elites = samples[elite_idx]

    ax.plot(xs, toy_return(xs), color="black", lw=2, label="score réel" if it == 0 else None)
    ax.scatter(samples, toy_return(samples), s=12, alpha=0.25, color="tab:gray")
    ax.scatter(elites, toy_return(elites), s=22, alpha=0.9, color="tab:orange", label="élites" if it == 0 else None)
    ax.axvline(mu, color="tab:blue", lw=2, label=r"$\mu$" if it == 0 else None)
    ax.set_title(f"Itération {it+1}\nmu={mu:.2f}, sigma={sigma:.2f}")
    ax.set_xlabel("action candidate")
    ax.grid(alpha=0.25)

    elite_mu = elites.mean()
    elite_sigma = elites.std() + 1e-6
    mu = (1 - alpha) * mu + alpha * elite_mu
    sigma = (1 - alpha) * sigma + alpha * elite_sigma

axes[0].set_ylabel("score imaginé")
axes[0].legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

**Lecture.** Les points gris sont les plans testés, les points orange sont les élites. À chaque
itération, la moyenne se rapproche de la zone rentable et l'écart-type diminue. Dans PETS, la variable
n'est pas une seule action scalaire, mais une séquence complète de taille $H \times A$.


## VI.4 — L'hypothèse : récompense connue sur l'état

Pour évaluer un plan, CEM doit calculer une récompense sur des états qui n'ont pas été observés dans
le vrai environnement : ce sont des états **prédits** par le modèle. Il faut donc disposer d'une
fonction :

$$r(s,a,s')$$

appelable sur les particules imaginées.

PETS suppose que cette fonction de récompense est connue. C'est souvent le cas dans les benchmarks de
contrôle : on connaît exactement ce qui est récompensé. Pour HalfCheetah, l'objectif est d'avancer
vite vers l'avant, tout en évitant des actions trop violentes.

On a volontairement créé l'environnement avec la position $x$ incluse dans l'observation, car la
récompense dépend de la variation de cette position. Pour HalfCheetah :

$$r(s, a, s') =
w_{\text{fwd}}\,\frac{x' - x}{\Delta t}
-
w_{\text{ctrl}}\,\lVert a\rVert^2$$

avec :

$$w_{\text{fwd}}=1,
\qquad
w_{\text{ctrl}}=0.1,
\qquad
\Delta t = 0.05$$

| Terme | Rôle |
|-------|------|
| $\frac{x'-x}{\Delta t}$ | vitesse avant du torso |
| $w_{\text{fwd}}$ | poids de la récompense d'avancement |
| $\lVert a\rVert^2$ | énergie approximative de l'action |
| $w_{\text{ctrl}}$ | poids du coût de contrôle |
| $\Delta t$ | durée physique d'un pas de simulation |

Cette récompense crée une tension utile : le planificateur veut avancer vite, mais pas en saturant
tous les moteurs sans discernement. Un plan qui fait gagner beaucoup de vitesse peut être rejeté s'il
dépense trop de couple.

On vérifiera que cette formule reproduit la récompense renvoyée par l'environnement. Si la récompense
était inconnue, il faudrait apprendre un modèle de récompense en plus du modèle de dynamique. PETS
dans sa forme classique évite cette complication : il apprend la dynamique, mais garde la fonction de
récompense analytique.

In [ ]:
# Fonction de récompense HalfCheetah (from scratch)

def halfcheetah_reward(obs, act, next_obs, dt=0.05, w_fwd=1.0, w_ctrl=0.1):
    # obs/next_obs : [..., 18], act : [..., 6]  (tenseurs float32)
    # Retourne récompense scalaire ou vecteur selon les dimensions leading
    x_velocity    = (next_obs[..., 0] - obs[..., 0]) / dt   # avance/recule
    forward_reward = w_fwd * x_velocity
    ctrl_cost      = w_ctrl * (act ** 2).sum(dim=-1)
    return forward_reward - ctrl_cost

# Mini-test : comparer avec la récompense de l'environnement réel

obs_env, _ = env.reset(seed=1)
rewards_env, rewards_fn = [], []
for _ in range(30):
    act = env.action_space.sample()
    next_obs_env, r_env, terminated, truncated, _ = env.step(act)
    obs_t      = torch.from_numpy(obs_env).float()
    act_t      = torch.from_numpy(act).float()
    next_obs_t = torch.from_numpy(next_obs_env).float()
    r_fn = float(halfcheetah_reward(obs_t, act_t, next_obs_t, dt=env.unwrapped.dt))
    rewards_env.append(r_env)
    rewards_fn.append(r_fn)
    obs_env = next_obs_env
    if terminated or truncated:
        obs_env, _ = env.reset()

max_diff = max(abs(a-b) for a,b in zip(rewards_env, rewards_fn))
print(f"Max |r_env - r_fn| sur 30 pas : {max_diff:.6f}  (atol=1e-3 attendu)")
assert max_diff < 1e-2, f"Récompense incorrecte : diff={max_diff:.6f}"
print("✓ halfcheetah_reward correspond à l'environnement")

## VI.X — Rendre CEM un peu plus prudent : le `risk_beta`

Dans la version de base, CEM classe chaque séquence d'actions par son **rendement moyen** sur les
particules :

$$
J(a_{0:H-1}) = \frac{1}{P}\sum_{p=1}^{P} R^{(p)}.
$$

C'est naturel, mais parfois trop optimiste. Si quelques particules prédisent un futur très favorable
à cause d'une erreur du modèle, elles peuvent tirer la moyenne vers le haut et faire sélectionner un
plan fragile. Sur HalfCheetah, c'est particulièrement visible : une petite surestimation de
$\Delta x$ devient une reward forward trop élevée, et CEM peut choisir des actions coûteuses qui
n'avancent pas vraiment dans l'environnement réel.

On ajoute donc une variante **risk-aware** : au lieu de maximiser seulement la moyenne, on pénalise
la dispersion du rendement entre particules :

$$
J_{\text{robuste}}(a_{0:H-1})
=
\underbrace{\mathbb{E}_{p}[R^{(p)}]}_{\text{rendement moyen}}
-
\beta_{\text{risk}}
\underbrace{\operatorname{Std}_{p}[R^{(p)}]}_{\text{désaccord / fragilité du plan}}.
$$

Le coefficient `risk_beta` règle le pessimisme :

| `risk_beta` | Effet |
|------------|-------|
| `0.0` | PETS standard : CEM optimise le rendement moyen |
| petit (`0.1–0.3`) | évite les plans très incertains sans bloquer l'exploration |
| grand (`>0.5`) | devient conservateur : moins d'actions risquées, parfois trop immobile |

Ce n'est pas une nouvelle méthode magique ; c'est un garde-fou pédagogique pour montrer une idée
importante en model-based RL : **un plan bon seulement dans quelques futurs optimistes n'est pas un bon
plan**. On préfère parfois un rendement imaginé un peu plus faible mais cohérent entre particules,
car il a plus de chances de survivre au passage dans le vrai simulateur.

In [ ]:
# ============================================================
# CEMPlanner — version risk-aware optionnelle
# ============================================================

class CEMPlanner:
    # Planificateur CEM : optimise une séquence d'actions par échantillonnage.
    # risk_beta > 0 pénalise les plans dont le return varie beaucoup entre particules.

    _MIN_STD = 1e-5

    def __init__(
        self,
        action_dim,
        horizon,
        population,
        elite_frac,
        iterations,
        alpha,
        action_low,
        action_high,
        risk_beta=0.0,
    ):
        self.action_dim = action_dim
        self.horizon = horizon
        self.population = population
        self.elite_frac = elite_frac
        self.iterations = iterations
        self.alpha = alpha
        self.action_low = action_low
        self.action_high = action_high
        self.risk_beta = float(risk_beta)
        self.n_elites = max(1, int(np.ceil(elite_frac * population)))
        self._init_std = ((action_high - action_low) / 2.0).clamp_min(1e-6)

    def plan(self, state, ensemble, reward_fn, n_particles, prev_mean=None):
        H, A = self.horizon, self.action_dim
        low, high = self.action_low, self.action_high

        if prev_mean is not None:
            mu = torch.cat([prev_mean[1:], torch.zeros(1, A)], dim=0)
        else:
            mu = torch.zeros(H, A)

        std = self._init_std.unsqueeze(0).expand(H, A).clone()

        for _ in range(self.iterations):
            noise = torch.randn(self.population, H, A)
            acts = (mu.unsqueeze(0) + std.unsqueeze(0) * noise).clamp(
                low.view(1, 1, A),
                high.view(1, 1, A),
            )

            scores = self._evaluate(acts, state, ensemble, reward_fn, n_particles)

            elite_idx = scores.topk(self.n_elites).indices
            elites = acts[elite_idx]
            e_mean = elites.mean(0)
            e_std = elites.std(0).clamp_min(self._MIN_STD)

            mu = (1 - self.alpha) * mu + self.alpha * e_mean
            std = (1 - self.alpha) * std + self.alpha * e_std

        mu = mu.clamp(low.view(1, A), high.view(1, A))
        first_action = mu[0].detach().cpu().numpy().astype(np.float32)
        return first_action, mu.detach()

    @torch.no_grad()
    def _evaluate(self, acts, state, ensemble, reward_fn, n_particles):
        N, H, A = acts.shape
        E = ensemble.n_members
        obs_dim = state.shape[0]
        B = N * n_particles

        states = state.unsqueeze(0).expand(B, obs_dim).clone()
        particle_member = torch.arange(B) % E
        acts_exp = acts.repeat_interleave(n_particles, dim=0)

        cumrew = torch.zeros(B)
        for t in range(H):
            act_t = acts_exp[:, t, :]
            next_states = ensemble.propagate(states, act_t, particle_member)
            rew_t = reward_fn(states, act_t, next_states)
            cumrew += rew_t
            states = next_states

        returns_by_particle = cumrew.view(N, n_particles)
        mean_return = returns_by_particle.mean(dim=1)

        if self.risk_beta <= 0.0 or n_particles <= 1:
            return mean_return

        # Pénalité pessimiste : on préfère un plan un peu moins brillant mais stable
        # à un plan qui ne marche que pour quelques particules optimistes.
        return_std = returns_by_particle.std(dim=1)
        return mean_return - self.risk_beta * return_std


print("CEMPlanner défini — version risk-aware optionnelle")

## VI.6 — Assembler modèle et planner dans `PETSAgent`

Le modèle probabiliste et CEM restent deux briques séparées : l'un représente les futurs possibles,
l'autre recherche une séquence d'actions. `PETSAgent` les rassemble avec le buffer réel et le
warm-start MPC. Il sait ajuster le modèle, planifier une action et mémoriser une transition ; la boucle
d'environnement reste visible dans `run_planning_episode`.

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef primary fill:none,stroke:#2563eb,stroke-width:2px
    classDef secondary fill:none,stroke:#d97706,stroke-width:2px
    env["environnement réel"]
    buffer["buffer de transitions réelles"]
    ensemble["ensemble probabiliste"]
    particles["particules TS∞"]
    cem["CEM + MPC"]
    action["première action"]
    env --> buffer --> ensemble
    ensemble --> particles --> cem --> action --> env
    class ensemble,particles primary
    class cem,action secondary
```

In [ ]:
class PETSAgent:
    def __init__(
        self, obs_dim, action_dim, action_low, action_high, dt,
        *, n_members, hidden_dim, n_layers, horizon, population,
        elite_frac, cem_iters, cem_alpha, n_particles, risk_beta=0.0,
    ):
        self.model = NormalizedEnsemble(
            obs_dim + action_dim, obs_dim,
            n_members=n_members, hidden_dim=hidden_dim, n_layers=n_layers,
        )
        self.planner = CEMPlanner(
            action_dim, horizon, population, elite_frac, cem_iters, cem_alpha,
            action_low, action_high, risk_beta=risk_beta,
        )
        self.dt = dt
        self.n_particles = n_particles
        self.buffer = ([], [], [])
        self.reset_planning()

    def reset_planning(self):
        self.previous_mean = None

    def reward(self, observation, action, next_observation):
        return halfcheetah_reward(observation, action, next_observation, dt=self.dt)

    def collect_warmup(self, env, n_steps, seed):
        self.buffer = collect_random_transitions(env, n_steps, seed)
        return len(self.buffer[0])

    def fit_model(self, *, steps, learning_rate, batch_size):
        inputs, targets = buffer_to_tensors(*self.buffer)
        nll_history = self.model.fit(
            inputs, targets, steps=steps, lr=learning_rate, batch_size=batch_size,
        )
        disagreement = self.model.disagreement(inputs[: min(500, len(inputs))])
        return float(np.mean(nll_history)), disagreement

    def select_action(self, observation):
        action, self.previous_mean = self.planner.plan(
            torch.as_tensor(observation, dtype=torch.float32),
            self.model, self.reward, self.n_particles, self.previous_mean,
        )
        return action

    def store_transition(self, observation, action, next_observation):
        self.buffer[0].append(observation.copy())
        self.buffer[1].append(action.copy())
        self.buffer[2].append(next_observation.copy())


print("PETSAgent défini : ensemble, buffer réel, CEM risk-aware optionnel")

In [ ]:
# Visualisation des itérations CEM : histogrammes des returns + décroissance de sigma

torch.manual_seed(0)
np.random.seed(0)

obs_plan, _ = env.reset(seed=3)
s_plan = torch.from_numpy(obs_plan).float()

# Paramètres CEM démo (rapides)
H_cem   = 12
N_cem   = 150
K_frac  = 0.1
I_cem   = 5
P_part  = 15

low_t  = torch.from_numpy(env.action_space.low).float()
high_t = torch.from_numpy(env.action_space.high).float()

planner_demo = CEMPlanner(
    action_dim=ACT_DIM, horizon=H_cem, population=N_cem,
    elite_frac=K_frac, iterations=I_cem, alpha=0.7,
    action_low=low_t, action_high=high_t
)

# Rejouer les itérations manuellement pour enregistrer les distributions
mu_cem = torch.zeros(H_cem, ACT_DIM)
std_cem = planner_demo._init_std.unsqueeze(0).expand(H_cem, ACT_DIM).clone()
returns_by_iter = []
sigma_by_iter   = []

for it in range(I_cem):
    noise = torch.randn(N_cem, H_cem, ACT_DIM)
    acts  = (mu_cem.unsqueeze(0) + std_cem.unsqueeze(0) * noise).clamp(
        low_t.view(1,1,ACT_DIM), high_t.view(1,1,ACT_DIM))
    with torch.no_grad():
        rets = planner_demo._evaluate(acts, s_plan, dyn,
                                      lambda o,a,no: halfcheetah_reward(o,a,no, dt=env.unwrapped.dt),
                                      n_particles=P_part)
    returns_by_iter.append(rets.numpy().copy())
    sigma_by_iter.append(float(std_cem.mean()))
    elite_idx = rets.topk(planner_demo.n_elites).indices
    elites    = acts[elite_idx]
    e_mean    = elites.mean(0)
    e_std     = elites.std(0).clamp_min(CEMPlanner._MIN_STD)
    mu_cem    = (1 - planner_demo.alpha)*mu_cem  + planner_demo.alpha*e_mean
    std_cem   = (1 - planner_demo.alpha)*std_cem + planner_demo.alpha*e_std

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

# Gauche : histogrammes des returns par itération
colors_it = plt.cm.Blues(np.linspace(0.3, 0.95, I_cem))
for i, rets_i in enumerate(returns_by_iter):
    axes[0].hist(rets_i, bins=20, alpha=0.6, color=colors_it[i], label=f"Iter {i+1}")
axes[0].set_xlabel("Return cumulé")
axes[0].set_ylabel("Compte")
axes[0].set_title("CEM : distribution des returns par itération")
axes[0].legend(fontsize=8)

# Droite : sigma moyen décroit
axes[1].plot(range(1, I_cem+1), sigma_by_iter, color="crimson", lw=2, marker="o")
axes[1].set_xlabel("Itération CEM")
axes[1].set_ylabel("Sigma moyen")
axes[1].set_title("CEM : resserrement de la recherche")
plt.tight_layout()
plt.show()

**Lecture.** À chaque itération, la masse des rendements se **décale vers la droite** : CEM concentre
les tirages sur des plans de mieux en mieux notés, pendant que $\sigma$ **rétrécit** — la recherche
passe d'exploratoire à précise. Quelques itérations suffisent (typiquement 5). On voit là toute la
philosophie : pas de gradient, juste « tirer, garder les meilleurs, resserrer », ce qui se marie
parfaitement avec un modèle qu'on ne sait qu'**échantillonner**.

In [ ]:
# Mini-test visuel CEM sur objectif jouet : action -> reward = action (1D)

torch.manual_seed(0)
rng = np.random.default_rng(0)

class TrivialEnsemble:
    # Modèle trivial : next_state = state + action (1D)
    n_members = 1

    def propagate(self, states, actions, model_idx):
        return states + actions

def trivial_reward(obs, act, next_obs):
    # La récompense croît avec l'action : le meilleur choix est donc +1.
    return act.sum(dim=-1)

trivial_env_ens = TrivialEnsemble()

planner_toy = CEMPlanner(
    action_dim=1,
    horizon=3,
    population=100,
    elite_frac=0.1,
    iterations=8,
    alpha=0.3,
    action_low=torch.tensor([-1.0]),
    action_high=torch.tensor([1.0]),
)

s_toy = torch.zeros(1)
first_act, mu_final = planner_toy.plan(
    s_toy,
    trivial_env_ens,
    trivial_reward,
    n_particles=1,
)

assert first_act[0] > 0.8, f"CEM n'a pas convergé vers +1 : {first_act[0]:.4f}"

# Rejouons une version instrumentée du même problème pour visualiser le resserrement de CEM.
mu = torch.zeros(planner_toy.horizon, planner_toy.action_dim)
sigma = torch.ones_like(mu)
elite_count = max(1, int(planner_toy.population * planner_toy.elite_frac))

trace = []
for _ in range(planner_toy.iterations):
    actions = mu + sigma * torch.randn(
        planner_toy.population,
        planner_toy.horizon,
        planner_toy.action_dim,
    )
    actions = torch.clamp(actions, planner_toy.action_low, planner_toy.action_high)

    # Ici le reward total vaut simplement la somme des actions sur l'horizon.
    returns = actions.sum(dim=(1, 2))
    elite_idx = torch.topk(returns, elite_count).indices
    elites = actions[elite_idx]

    trace.append(
        {
            "mu0": float(mu[0, 0]),
            "sigma0": float(sigma[0, 0]),
            "best_return": float(returns.max()),
            "elite_mean0": float(elites[:, 0, 0].mean()),
            "samples0": actions[:, 0, 0].detach().numpy(),
            "returns": returns.detach().numpy(),
        }
    )

    elite_mu = elites.mean(dim=0)
    elite_sigma = elites.std(dim=0).clamp_min(1e-3)
    mu = (1 - planner_toy.alpha) * mu + planner_toy.alpha * elite_mu
    sigma = (1 - planner_toy.alpha) * sigma + planner_toy.alpha * elite_sigma

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

axes[0].plot([t["mu0"] for t in trace], marker="o", lw=2, label=r"$\mu_0$")
axes[0].plot([t["elite_mean0"] for t in trace], marker="o", lw=2, label="moyenne élite")
axes[0].axhline(1.0, color="black", ls="--", lw=1, label="borne optimale +1")
axes[0].set_title("La moyenne CEM se déplace vers +1")
axes[0].set_xlabel("itération CEM")
axes[0].set_ylabel("première action")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot([t["sigma0"] for t in trace], marker="o", lw=2, color="tab:orange")
axes[1].set_title("La distribution se resserre")
axes[1].set_xlabel("itération CEM")
axes[1].set_ylabel(r"$\sigma_0$")
axes[1].grid(alpha=0.25)

for idx in [0, 2, 5, 7]:
    axes[2].hist(
        trace[idx]["samples0"],
        bins=20,
        range=(-1, 1),
        alpha=0.35,
        label=f"it. {idx + 1}",
    )
axes[2].axvline(1.0, color="black", ls="--", lw=1)
axes[2].set_title("Échantillons de la première action")
axes[2].set_xlabel(r"$a_0$")
axes[2].set_ylabel("nombre")
axes[2].legend()
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print(f"Action finale CEM : {first_act[0]:.4f}  (objectif jouet : proche de +1.0)")
print(f"Moyenne finale a0 : {float(mu_final[0, 0]):.4f}")


**Lecture.** Sur cet objectif jouet, la meilleure action est trivialement la borne haute `+1`. Le test vérifie que le planner trouve bien cette direction. Les courbes montrent le mécanisme interne : la moyenne de la distribution CEM se décale vers les élites, puis l’écart-type diminue pour concentrer les prochains essais autour des bonnes actions.

## VI.5 — Le warm-start : ne pas repartir de zéro

Replanifier intégralement à chaque pas serait du gaspillage. Entre deux pas consécutifs, le monde n'a
pas changé arbitrairement : on a exécuté la première action, observé un nouvel état proche de ce qui
était prévu, puis on doit choisir la suite. Le plan optimal d'aujourd'hui ressemble donc souvent au
plan calculé juste avant.

Supposons qu'à l'instant $t$, CEM trouve le plan :

$$[a_t,\ a_{t+1},\ a_{t+2},\dots,\ a_{t+H-1}]$$

On exécute seulement $a_t$. À l'instant suivant, il est naturel de réutiliser la suite :

$$[a_{t+1},\ a_{t+2},\dots,\ a_{t+H-1},\ 0]$$

L'action prévue pour « demain » devient l'estimation initiale d'« aujourd'hui », et on complète la fin
avec une action neutre. C'est le **warm-start**.

Intuitivement, c'est comme recalculer un itinéraire GPS après avoir avancé de cent mètres. On ne
repart pas d'une carte blanche : l'itinéraire précédent reste une très bonne hypothèse de départ, mais
on le corrige avec la nouvelle position réelle.

Dans CEM, cela veut dire que la moyenne initiale $\mu$ de la distribution de plans n'est pas remise à
zéro. Elle est initialisée avec le plan précédent décalé. La recherche commence donc déjà près d'une
solution plausible, et les échantillons servent surtout à ajuster le plan plutôt qu'à le redécouvrir.

Ce petit détail d'ingénierie compte beaucoup :

- moins d'itérations CEM nécessaires ;
- actions plus stables d'un pas à l'autre ;
- meilleure utilisation du budget de calcul ;
- comportement MPC moins erratique.

Sans warm-start, chaque pas redémarre comme si l'on ne savait rien. Avec warm-start, PETS garde une
mémoire courte du plan précédent, tout en restant capable de le corriger dès que la vraie observation
s'écarte de ce qui était imaginé.

Toutes les briques sont là : ensemble probabiliste (incertitude calibrée), trajectory sampling
(propagation), récompense connue, CEM + MPC (planning). Assemblons la boucle PETS complète.

---
# Partie VII — La boucle PETS complète

## VII.1 — L'algorithme, en entier

On peut maintenant assembler toutes les briques. PETS alterne entre deux mondes :

- le **monde réel**, où l'on collecte peu de transitions parce qu'elles coûtent cher ;
- le **monde imaginé**, où l'on évalue beaucoup de plans dans l'ensemble de modèles.

La structure générale est simple : apprendre un modèle avec les données disponibles, planifier dans ce
modèle, exécuter seulement les premières actions dans l'environnement, puis enrichir le dataset.

$$
\boxed{
\begin{aligned}
&\textbf{Algorithme PETS}\\[2mm]
&\textbf{Entrées : }
\text{ environnement } \mathcal{E},\ \text{ensemble probabiliste } \{f_b\}_{b=1}^{B},\
\text{ horizon } H,\ \text{nombre de particules } P.\\[1mm]
&\textbf{Initialiser } \mathcal{D}
\text{ avec quelques transitions aléatoires } (s_t, a_t, s_{t+1}).\\[1mm]
&\textbf{Pour } i = 1,2,\dots,N_{\text{iter}} \textbf{ faire :}\\
&\quad \text{1. Entraîner chaque membre } f_b
\text{ sur } \mathcal{D}
\text{ pour prédire } \Delta s = s_{t+1}-s_t.\\
&\quad \text{2. Réinitialiser le warm-start du plan : } \bar{a}_{0:H-1} \leftarrow 0.\\
&\quad \text{3. Réinitialiser l'environnement réel et observer } s_0.\\
&\quad \textbf{Pour } t = 0,1,\dots,T-1 \textbf{ faire :}\\
&\quad\quad \text{a. Résoudre par CEM :}\\
&\quad\quad\quad
a_{t:t+H-1}^{\star}
\approx
\arg\max_{a_{t:t+H-1}}
\frac{1}{P}\sum_{p=1}^{P}\sum_{h=0}^{H-1}
r\!\left(s_{t+h}^{(p)}, a_{t+h}, s_{t+h+1}^{(p)}\right),\\
&\quad\quad\quad
\text{où les particules sont propagées par TS}\infty
\text{ dans l'ensemble } \{f_b\}_{b=1}^{B}.\\
&\quad\quad \text{b. Exécuter seulement la première action } a_t^{\star}
\text{ dans le vrai environnement.}\\
&\quad\quad \text{c. Observer } s_{t+1}, r_t
\text{ et ajouter } (s_t, a_t^{\star}, s_{t+1}) \text{ à } \mathcal{D}.\\
&\quad\quad \text{d. Décaler le plan pour le warm-start suivant :}\\
&\quad\quad\quad
\bar{a}_{0:H-2} \leftarrow a_{t+1:t+H-1}^{\star},
\qquad
\bar{a}_{H-1} \leftarrow 0.\\
&\quad \textbf{Fin pour}\\
&\quad \text{4. Journaliser : NLL du modèle, désaccord d'ensemble, rendement réel.}\\
&\textbf{Fin pour}
\end{aligned}
}
$$

La ligne importante est l'étape **3a** : CEM ne cherche pas une action isolée, mais une séquence
d'actions. Cette séquence est évaluée en imaginant plusieurs futurs avec les particules. Le score d'un
plan est donc une récompense moyenne sur un faisceau de trajectoires plausibles, pas sur une seule
prédiction déterministe.

Ensuite, l'étape **3b** applique la logique MPC : on ne joue que la première action du plan. Les autres
actions ne sont pas inutiles ; elles ont servi à juger si la première action menait vers un futur
prometteur. Mais on ne les exécute pas aveuglément, parce que la prochaine observation réelle donnera
une information plus fiable que nos prédictions lointaines.

Une seule étape touche le vrai monde : **exécuter $a_t^\star$ dans l'environnement**. Tout le reste
se passe dans le modèle :

- l'entraînement de l'ensemble utilise les transitions déjà collectées ;
- CEM teste des centaines de séquences candidates ;
- chaque séquence est évaluée sur plusieurs particules ;
- les rollouts imaginés ne coûtent aucune interaction réelle.

C'est la raison de l'efficacité-échantillon de PETS. Le modèle devient une sorte de simulateur local :
pas parfait, mais suffisamment utile pour remplacer une grande partie des essais réels par des essais
imaginés.

In [ ]:
# Helpers de collecte et d'épisode réel pour PETS

def reward_parts_numpy(obs, action, next_obs, dt, ctrl_weight=0.1):
    # Même décomposition que HalfCheetah : avance moins coût de contrôle.
    forward = (float(next_obs[0]) - float(obs[0])) / dt
    ctrl = ctrl_weight * float(np.square(action).sum())
    return forward, ctrl


def collect_random_transitions(env, n_steps, seed):
    # Buffer initial : transitions officielles, actions uniformes dans l'espace d'action.
    buf_obs, buf_act, buf_nobs = [], [], []
    np.random.seed(seed)
    torch.manual_seed(seed)
    env.action_space.seed(seed)
    obs, _ = env.reset(seed=seed)

    for k in range(n_steps):
        action = env.action_space.sample()
        next_obs, _, terminated, truncated, _ = env.step(action)
        buf_obs.append(obs.copy())
        buf_act.append(action.copy())
        buf_nobs.append(next_obs.copy())
        obs = next_obs
        if terminated or truncated:
            obs, _ = env.reset(seed=seed + k + 1)
    return buf_obs, buf_act, buf_nobs


def buffer_to_tensors(buf_obs, buf_act, buf_nobs):
    # X=[s,a], Y=Delta s : le format d'entraînement du modèle de dynamique.
    X = torch.from_numpy(np.concatenate([buf_obs, buf_act], axis=1).astype(np.float32))
    Y = torch.from_numpy((np.array(buf_nobs) - np.array(buf_obs)).astype(np.float32))
    return X, Y


def run_planning_episode(env, agent, *, seed, max_steps, collect=True):
    # L'environnement reste ici : l'agent ne reçoit que l'observation courante.
    dt = env.unwrapped.dt
    obs, _ = env.reset(seed=seed)
    agent.reset_planning()
    x0 = float(obs[0])
    total_return = 0.0
    forwards, ctrls, action_norms = [], [], []
    trajectory_x = [x0]

    for _ in range(max_steps):
        action = agent.select_action(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        forward, ctrl = reward_parts_numpy(obs, action, next_obs, dt)
        if collect:
            agent.store_transition(obs, action, next_obs)
        forwards.append(forward); ctrls.append(ctrl)
        action_norms.append(float(np.linalg.norm(action)))
        total_return += float(reward)
        obs = next_obs
        trajectory_x.append(float(obs[0]))
        if terminated or truncated:
            break

    steps = max(1, len(forwards))
    distance = float(trajectory_x[-1]) - x0
    return {
        "return": total_return,
        "distance": distance,
        "mean_velocity": distance / (steps * dt),
        "forward_reward": float(np.sum(forwards)),
        "ctrl_cost": float(np.sum(ctrls)),
        "action_norm": float(np.mean(action_norms)),
        "x": np.asarray(trajectory_x),
    }


In [ ]:
# ============================================================
# Boucle PETS complète — réglages notebook plus démonstratifs
# ============================================================

def run_pets(
    env,
    # Ensemble
    n_members      = 5,
    hidden_dim     = 128,
    n_layers       = 3,
    fit_steps      = 250,       # pas Adam par itération PETS
    fit_lr         = 1e-3,
    fit_bs         = 256,
    # CEM
    horizon        = 20,
    population     = 220,
    elite_frac     = 0.1,
    cem_iters      = 5,
    cem_alpha      = 0.7,       # poids donné aux élites ; 0.1 était trop timide ici
    n_particles    = 12,
    # Boucle
    warmup_steps   = 1000,      # transitions aléatoires initiales
    pets_iters     = 8,         # itérations PETS
    max_steps      = 250,       # longueur max d'un épisode réel
    seed           = 0,
    risk_beta     = 0.25,
):
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    agent = PETSAgent(
        obs_dim, act_dim,
        torch.from_numpy(env.action_space.low).float(),
        torch.from_numpy(env.action_space.high).float(),
        env.unwrapped.dt,
        n_members=n_members, hidden_dim=hidden_dim, n_layers=n_layers,
        horizon=horizon, population=population, elite_frac=elite_frac,
        cem_iters=cem_iters, cem_alpha=cem_alpha, n_particles=n_particles,
        risk_beta=risk_beta
    )

    warmup_size = agent.collect_warmup(env, warmup_steps, seed)
    print(f"Warmup : {warmup_size} transitions collectées")

    logs = {
        "nll": [], "disagreement": [], "returns": [], "distance": [],
        "mean_velocity": [], "forward_reward": [], "ctrl_cost": [], "action_norm": [],
        "buffer_size": [],
    }

    for it in range(pets_iters):
        # 1) Apprendre le modèle sur toutes les transitions réelles collectées jusque-là.
        mean_nll, disagreement = agent.fit_model(
            steps=fit_steps, learning_rate=fit_lr, batch_size=fit_bs,
        )

        # 2) Contrôler un vrai épisode avec MPC+CEM, puis enrichir le buffer.
        stats = run_planning_episode(
            env, agent, seed=seed + it + 1, max_steps=max_steps, collect=True,
        )

        logs["nll"].append(mean_nll)
        logs["disagreement"].append(disagreement)
        logs["returns"].append(stats["return"])
        logs["distance"].append(stats["distance"])
        logs["mean_velocity"].append(stats["mean_velocity"])
        logs["forward_reward"].append(stats["forward_reward"])
        logs["ctrl_cost"].append(stats["ctrl_cost"])
        logs["action_norm"].append(stats["action_norm"])
        logs["buffer_size"].append(len(agent.buffer[0]))

        print(
            f"[PETS iter {it+1}/{pets_iters}]  "
            f"NLL={mean_nll:.3f}  désaccord={disagreement:.4f}  "
            f"retour={stats['return']:+.1f}  Δx={stats['distance']:+.2f}  "
            f"v={stats['mean_velocity']:+.3f}  |a|={stats['action_norm']:.2f}  "
            f"buf={len(agent.buffer[0])}"
        )

    return {
        "logs": logs,
        "agent": agent,
        "model": agent.model,
        "planner": agent.planner,
        "buffer": agent.buffer,
        "config": {
            "horizon": horizon,
            "population": population,
            "cem_iters": cem_iters,
            "cem_alpha": cem_alpha,
            "n_particles": n_particles,
            "max_steps": max_steps,
        },
    }

print("run_pets défini — prêt à lancer la démo renforcée")

In [ ]:
# Configuration du mini-training PETS.
# Version plus prudente : horizon plus court, plus de particules, CEM plus robuste.
PETS_DEMO_CONFIG = dict(
    n_members=5,
    hidden_dim=160,
    n_layers=3,
    fit_steps=400,
    fit_lr=1e-3,
    fit_bs=256,
    horizon=12,
    population=320,
    elite_frac=0.1,
    cem_iters=6,
    cem_alpha=0.6,
    n_particles=24,
    risk_beta=0.35,
    warmup_steps=1500,
    pets_iters=8,
    max_steps=250,
    seed=0,
)

np.random.seed(0)
torch.manual_seed(0)

In [ ]:
# Lancer la démo PETS renforcée
# Sur une machine CPU, cette cellule peut prendre plusieurs minutes : c'est normal,
# PETS planifie par CEM à chaque pas réel.


t0 = time.time()
pets_result = run_pets(env, **PETS_DEMO_CONFIG)
logs = pets_result["logs"]
pets_agent_final = pets_result["agent"]
dyn_final = pets_result["model"]
planner_final = pets_result["planner"]
buf_obs_f, buf_act_f, buf_nobs_f = pets_result["buffer"]

log_nll = logs["nll"]
log_disagree = logs["disagreement"]
log_returns = logs["returns"]
log_distance = logs["distance"]
print(f"\nDémo terminée en {time.time()-t0:.1f}s")


In [ ]:
# Diagnostics : modèle, contrôle réel, et baselines multi-seed


def rollout_baseline(env_id, policy, n_episodes=5, max_steps=250, seed=700):
    eval_env = gym.make(env_id, exclude_current_positions_from_observation=False)
    rows = []
    try:
        for ep in range(n_episodes):
            obs, _ = eval_env.reset(seed=seed + ep)
            eval_env.action_space.seed(seed + ep)
            x0 = float(obs[0])
            total_return, action_norms = 0.0, []
            for _ in range(max_steps):
                if policy == "zero":
                    action = np.zeros(eval_env.action_space.shape, dtype=np.float32)
                elif policy == "random":
                    action = eval_env.action_space.sample()
                else:
                    raise ValueError(policy)
                obs, reward, terminated, truncated, _ = eval_env.step(action)
                total_return += float(reward)
                action_norms.append(float(np.linalg.norm(action)))
                if terminated or truncated:
                    break
            rows.append({
                "return": total_return,
                "distance": float(obs[0]) - x0,
                "action_norm": float(np.mean(action_norms)),
            })
    finally:
        eval_env.close()
    return rows


def summarize_rows(rows):
    return {
        key: (float(np.mean([r[key] for r in rows])), float(np.std([r[key] for r in rows])))
        for key in rows[0]
    }

baseline_zero = summarize_rows(rollout_baseline("HalfCheetah-v5", "zero"))
baseline_random = summarize_rows(rollout_baseline("HalfCheetah-v5", "random"))

fig, axes = plt.subplots(2, 3, figsize=(13, 6.2))
iters = np.arange(1, len(logs["nll"]) + 1)

axes[0, 0].plot(iters, logs["nll"], color="steelblue", lw=2, marker="o")
axes[0, 0].set_title("NLL modèle (↓)")
axes[0, 0].set_xlabel("Itération PETS")
axes[0, 0].set_ylabel("NLL normalisée")

axes[0, 1].plot(iters, logs["disagreement"], color="darkorange", lw=2, marker="s")
axes[0, 1].set_title("Désaccord ensemble")
axes[0, 1].set_xlabel("Itération PETS")
axes[0, 1].set_ylabel("std moyenne")

axes[0, 2].plot(iters, logs["buffer_size"], color="slategray", lw=2, marker="^")
axes[0, 2].set_title("Taille du buffer réel")
axes[0, 2].set_xlabel("Itération PETS")
axes[0, 2].set_ylabel("transitions")

axes[1, 0].bar(iters, logs["returns"], color="forestgreen", alpha=0.82, label="PETS")
axes[1, 0].axhline(baseline_zero["return"][0], color="black", ls="--", lw=2, label="zéro")
axes[1, 0].axhline(baseline_random["return"][0], color="crimson", ls=":", lw=2, label="random")
axes[1, 0].set_title("Retour réel")
axes[1, 0].set_xlabel("Itération PETS")
axes[1, 0].set_ylabel("reward")
axes[1, 0].legend(fontsize=8)

axes[1, 1].bar(iters, logs["distance"], color="seagreen", alpha=0.82, label="PETS")
axes[1, 1].axhline(baseline_zero["distance"][0], color="black", ls="--", lw=2, label="zéro")
axes[1, 1].axhline(baseline_random["distance"][0], color="crimson", ls=":", lw=2, label="random")
axes[1, 1].set_title("Distance avancée Δx")
axes[1, 1].set_xlabel("Itération PETS")
axes[1, 1].set_ylabel("Δ position x")

axes[1, 2].plot(iters, logs["mean_velocity"], color="purple", lw=2, marker="o", label="vitesse")
axes[1, 2].plot(iters, logs["action_norm"], color="tab:brown", lw=2, marker="s", label=r"$||a||$")
axes[1, 2].set_title("Comportement du contrôleur")
axes[1, 2].set_xlabel("Itération PETS")
axes[1, 2].legend(fontsize=8)

for ax in axes.ravel():
    ax.grid(alpha=0.25)

fig.suptitle("Diagnostics PETS — modèle + contrôle réel", fontsize=13)
plt.tight_layout()
plt.show()

print("Baselines sur 5 seeds, même horizon :")
for name, stats in [("zéro", baseline_zero), ("random", baseline_random)]:
    print(
        f"  {name:6s} | retour {stats['return'][0]:+7.1f} ± {stats['return'][1]:5.1f} "
        f"| Δx {stats['distance'][0]:+6.2f} ± {stats['distance'][1]:4.2f} "
        f"| |a| {stats['action_norm'][0]:.2f}"
    )
print(
    f"Dernière itération PETS | retour {logs['returns'][-1]:+7.1f} "
    f"| Δx {logs['distance'][-1]:+6.2f} "
    f"| v {logs['mean_velocity'][-1]:+.3f} "
    f"| |a| {logs['action_norm'][-1]:.2f}"
)

**Lecture des diagnostics.** La NLL indique si l'ensemble apprend la dynamique ; les courbes de contrôle disent si cette dynamique devient utile pour agir. On compare maintenant PETS à deux références : `random`, qui explore mais paie un gros coût de contrôle, et `zero`, qui ne bouge presque pas mais évite ce coût. C'est important : battre `random` en reward ne suffit pas sur HalfCheetah, car une politique immobile peut déjà faire un retour proche de zéro. Le signal recherché est donc plus exigeant : **avancer davantage que zéro**, avec une vitesse positive et des actions non triviales mais pas saturées. Sur ce budget notebook, on attend un début de locomotion, pas une maîtrise comparable à SAC/TD3 entraînés longtemps.

In [ ]:
# Diagnostic modèle : erreur one-step sur Δx, vitesse et reward

def diagnose_one_step_reward_model(agent, n=500, seed=0):
    inputs, targets = buffer_to_tensors(*agent.buffer)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(inputs), size=min(n, len(inputs)), replace=False)

    X = inputs[idx]
    Y = targets[idx]

    obs = X[:, :OBS_DIM]
    act = X[:, OBS_DIM:]
    true_next = obs + Y

    with torch.no_grad():
        means, vars_ = agent.model.predict(X)
        pred_delta = means.mean(dim=0)
        pred_next = obs + pred_delta

        true_reward = halfcheetah_reward(obs, act, true_next, dt=agent.dt)
        pred_reward = halfcheetah_reward(obs, act, pred_next, dt=agent.dt)

        true_dx = true_next[:, 0] - obs[:, 0]
        pred_dx = pred_next[:, 0] - obs[:, 0]

        true_v = true_dx / agent.dt
        pred_v = pred_dx / agent.dt

    reward_err = (pred_reward - true_reward).detach().cpu().numpy()
    dx_err = (pred_dx - true_dx).detach().cpu().numpy()
    v_err = (pred_v - true_v).detach().cpu().numpy()

    print("Diagnostic one-step du modèle final")
    print(f"  reward MAE : {np.mean(np.abs(reward_err)):.4f}")
    print(f"  Δx MAE     : {np.mean(np.abs(dx_err)):.5f}")
    print(f"  vitesse MAE: {np.mean(np.abs(v_err)):.4f}")
    print(f"  reward bias: {np.mean(reward_err):+.4f}")
    print(f"  vitesse bias: {np.mean(v_err):+.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

    axes[0].scatter(true_reward.numpy(), pred_reward.numpy(), s=10, alpha=0.35)
    lo = min(float(true_reward.min()), float(pred_reward.min()))
    hi = max(float(true_reward.max()), float(pred_reward.max()))
    axes[0].plot([lo, hi], [lo, hi], "k--", lw=1)
    axes[0].set_title("Reward one-step : vraie vs prédite")
    axes[0].set_xlabel("reward vraie")
    axes[0].set_ylabel("reward prédite")

    axes[1].hist(v_err, bins=40, color="steelblue", alpha=0.8)
    axes[1].axvline(0.0, color="black", lw=1)
    axes[1].set_title("Erreur de vitesse forward")
    axes[1].set_xlabel(r"$\hat v_x - v_x$")

    axes[2].scatter(true_v.numpy(), pred_v.numpy(), s=10, alpha=0.35, color="darkorange")
    lo = min(float(true_v.min()), float(pred_v.min()))
    hi = max(float(true_v.max()), float(pred_v.max()))
    axes[2].plot([lo, hi], [lo, hi], "k--", lw=1)
    axes[2].set_title("Vitesse forward : vraie vs prédite")
    axes[2].set_xlabel("vitesse vraie")
    axes[2].set_ylabel("vitesse prédite")

    for ax in axes:
        ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()


diagnose_one_step_reward_model(pets_agent_final)

## VII.2 — Évaluation réelle et démonstration visuelle

Les courbes d'entraînement sont utiles, mais elles peuvent mentir si on ne regarde qu'une seule trajectoire. On ajoute donc une petite évaluation en conditions réelles : nouveaux resets officiels de `HalfCheetah-v5`, même planner MPC+CEM, pas de transition ajoutée au buffer pendant l'évaluation.

Cette évaluation reste courte parce que PETS réfléchit à chaque pas. Elle sert à répondre à une question simple : *le contrôleur final fait-il mieux qu'une action nulle ou qu'un comportement aléatoire sur des épisodes jamais vus ?*

In [ ]:
# Évaluation finale sur plusieurs épisodes réels.
# Même horizon que les baselines pour comparer proprement les barres.
PETS_EVAL_EPISODES = 5
PETS_EVAL_STEPS = 250

In [ ]:
# Évaluation réelle du contrôleur PETS final

eval_env = gym.make("HalfCheetah-v5", exclude_current_positions_from_observation=False)
try:
    pets_eval_rows = [
        run_planning_episode(
            eval_env,
            pets_agent_final,
            seed=900 + ep,
            max_steps=PETS_EVAL_STEPS,
            collect=False,
        )
        for ep in range(PETS_EVAL_EPISODES)
    ]
finally:
    eval_env.close()

pets_eval = summarize_rows(pets_eval_rows)

# Baselines recalculées sur exactement le même horizon que PETS.
baseline_zero_eval = summarize_rows(
    rollout_baseline("HalfCheetah-v5", "zero", n_episodes=PETS_EVAL_EPISODES, max_steps=PETS_EVAL_STEPS, seed=900)
)
baseline_random_eval = summarize_rows(
    rollout_baseline("HalfCheetah-v5", "random", n_episodes=PETS_EVAL_EPISODES, max_steps=PETS_EVAL_STEPS, seed=900)
)

print("Évaluation finale sur resets officiels, même horizon :")
for name, stats in [("zéro", baseline_zero_eval), ("random", baseline_random_eval), ("PETS", pets_eval)]:
    print(
        f"  {name:6s} | retour {stats['return'][0]:+7.1f} ± {stats['return'][1]:5.1f} "
        f"| Δx {stats['distance'][0]:+6.2f} ± {stats['distance'][1]:4.2f} "
        f"| v {stats.get('mean_velocity', (float('nan'), 0.0))[0]:+.3f} "
        f"| |a| {stats['action_norm'][0]:.2f}"
    )

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
labels = ["zéro", "random", "PETS"]

returns_mean = [baseline_zero_eval["return"][0], baseline_random_eval["return"][0], pets_eval["return"][0]]
returns_std = [baseline_zero_eval["return"][1], baseline_random_eval["return"][1], pets_eval["return"][1]]

distance_mean = [baseline_zero_eval["distance"][0], baseline_random_eval["distance"][0], pets_eval["distance"][0]]
distance_std = [baseline_zero_eval["distance"][1], baseline_random_eval["distance"][1], pets_eval["distance"][1]]

action_mean = [baseline_zero_eval["action_norm"][0], baseline_random_eval["action_norm"][0], pets_eval["action_norm"][0]]
action_std = [baseline_zero_eval["action_norm"][1], baseline_random_eval["action_norm"][1], pets_eval["action_norm"][1]]

colors = ["gray", "crimson", "forestgreen"]

axes[0].bar(labels, returns_mean, yerr=returns_std, capsize=4, color=colors, alpha=0.8)
axes[0].set_title("Retour réel moyen")
axes[0].set_ylabel("reward")
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(labels, distance_mean, yerr=distance_std, capsize=4, color=colors, alpha=0.8)
axes[1].axhline(0.0, color="black", lw=1)
axes[1].set_title("Distance avancée moyenne")
axes[1].set_ylabel("Δx")
axes[1].grid(axis="y", alpha=0.25)

axes[2].bar(labels, action_mean, yerr=action_std, capsize=4, color=colors, alpha=0.8)
axes[2].set_title("Amplitude d'action")
axes[2].set_ylabel(r"$\|a\|$")
axes[2].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

**Lecture.** Le retour et la distance répondent à deux questions complémentaires : PETS reçoit-il réellement davantage de récompense, et ce gain correspond-il bien à un déplacement vers l'avant plutôt qu'à un artefact du coût de contrôle ? Les barres d'erreur viennent de resets officiels distincts. Une amélioration accompagnée d'une forte variance reste un signal de mini-training, pas une résolution robuste de HalfCheetah ; le benchmark complet demanderait davantage de données, de seeds et un budget CEM plus élevé.

In [ ]:
# Diagnostic : rendement imaginé du plan choisi vs rendement réel obtenu

def evaluate_first_plan_imagined_vs_real(agent, seed=123, horizon=None):
    probe_env = gym.make("HalfCheetah-v5", exclude_current_positions_from_observation=False)
    try:
        obs, _ = probe_env.reset(seed=seed)
        agent.reset_planning()

        state_t = torch.as_tensor(obs, dtype=torch.float32)
        _, mean_plan = agent.planner.plan(
            state_t,
            agent.model,
            agent.reward,
            agent.n_particles,
            prev_mean=None,
        )

        if horizon is None:
            horizon = mean_plan.shape[0]
        actions = mean_plan[:horizon]

        # Rendement imaginé : moyenne sur particules TS∞.
        P = agent.n_particles
        states = state_t.unsqueeze(0).expand(P, -1).clone()
        model_idx = torch.arange(P) % agent.model.n_members
        imagined_rewards = []

        with torch.no_grad():
            for t in range(horizon):
                act_t = actions[t].unsqueeze(0).expand(P, -1)
                next_states = agent.model.propagate(states, act_t, model_idx)
                rew_t = agent.reward(states, act_t, next_states)
                imagined_rewards.append(float(rew_t.mean().item()))
                states = next_states

        imagined_return = float(np.sum(imagined_rewards))

        # Rendement réel : mêmes actions, même état initial.
        obs_real, _ = probe_env.reset(seed=seed)
        real_rewards = []
        x0 = float(obs_real[0])

        for t in range(horizon):
            action = actions[t].detach().cpu().numpy().astype(np.float32)
            next_obs, reward, terminated, truncated, _ = probe_env.step(action)
            real_rewards.append(float(reward))
            obs_real = next_obs
            if terminated or truncated:
                break

        real_return = float(np.sum(real_rewards))
        real_distance = float(obs_real[0]) - x0

    finally:
        probe_env.close()

    return {
        "imagined_return": imagined_return,
        "real_return": real_return,
        "real_distance": real_distance,
        "imagined_rewards": imagined_rewards,
        "real_rewards": real_rewards,
    }


plan_gap = evaluate_first_plan_imagined_vs_real(pets_agent_final, seed=123, horizon=PETS_DEMO_CONFIG["horizon"])

print("Plan choisi par CEM : imagination vs réalité")
print(f"  return imaginé : {plan_gap['imagined_return']:+.2f}")
print(f"  return réel    : {plan_gap['real_return']:+.2f}")
print(f"  gap modèle     : {plan_gap['imagined_return'] - plan_gap['real_return']:+.2f}")
print(f"  Δx réel        : {plan_gap['real_distance']:+.2f}")

fig, ax = plt.subplots(figsize=(8, 3.6))
t_im = np.arange(1, len(plan_gap["imagined_rewards"]) + 1)
t_re = np.arange(1, len(plan_gap["real_rewards"]) + 1)

ax.plot(t_im, np.cumsum(plan_gap["imagined_rewards"]), marker="o", lw=2, label="cumul imaginé")
ax.plot(t_re, np.cumsum(plan_gap["real_rewards"]), marker="s", lw=2, label="cumul réel")
ax.axhline(0.0, color="black", lw=1)
ax.set_title("Même plan d'actions : reward imaginée vs reward réelle")
ax.set_xlabel("pas dans l'horizon CEM")
ax.set_ylabel("reward cumulée")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Ce diagnostic sépare deux causes possibles d'échec. Si le rendement imaginé et le
rendement réel sont tous les deux faibles, le planner ne trouve simplement pas de bon plan. Si le
rendement imaginé est élevé mais le rendement réel chute, le planner exploite une erreur du modèle :
c'est le biais de modèle classique de PETS. Dans ce second cas, augmenter seulement CEM ne suffit pas ;
il faut améliorer le modèle, réduire l'horizon, augmenter les particules, ou collecter davantage de
données dans les régions visitées par le planner.

In [ ]:
DEMO_STEPS = 200  # Longueur du replay vidéo final.

In [ ]:
# Démonstration visuelle : replay vidéo notebook, sans fenêtre une fenêtre locale.
def evaluate_and_display_agent(agent, label="PETS", seed=1234, max_steps=200, video_root="videos/11_pets_halfcheetah"):
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    demo_env = gym.make(
        "HalfCheetah-v5",
        render_mode="rgb_array",
        exclude_current_positions_from_observation=False,
    )
    demo_env = RecordVideo(
        demo_env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == 0,
        name_prefix=safe_label,
        disable_logger=True,
    )

    try:
        stats = run_planning_episode(
            demo_env,
            agent,
            seed=seed,
            max_steps=max_steps,
            collect=False,
        )
    finally:
        demo_env.close()

    print(
        f"Démo {label} | retour={stats['return']:+.1f} "
        f"| Δx={stats['distance']:+.2f} "
        f"| v={stats['mean_velocity']:+.3f} "
        f"| |a|={stats['action_norm']:.2f}"
    )

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=640))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return stats


try:
    pets_demo_stats = evaluate_and_display_agent(pets_agent_final, max_steps=DEMO_STEPS)
except Exception as exc:
    print(f"Vidéo non disponible dans cet environnement : {type(exc).__name__}: {exc}")


**Lecture.** La vidéo est le test de bon sens final : on doit voir si le planificateur produit seulement des actions prudentes proches de zéro, ou s'il commence réellement à pousser le corps vers l'avant. Le mode `human` est volontairement optionnel, car il ouvre une fenêtre locale et peut bloquer sur certains environnements distants. La vidéo `rgb_array` garde une trace inspectable directement dans le notebook.

## VII.3 — Imagination vs réalité

Dernier diagnostic : PETS contrôle par imagination, donc il faut vérifier à quel point cette imagination reste alignée avec le vrai simulateur. On prend un même état initial, une séquence d'actions planifiée par le modèle final, puis on compare la trajectoire prédite par les particules à la trajectoire réellement obtenue dans MuJoCo.

In [ ]:
# Imagination vs réalité : trajectoire prédite vs trajectoire vraie

torch.manual_seed(0)
np.random.seed(0)

H_cmp = 10
P_cmp = PETS_DEMO_CONFIG["n_particles"]
dim_cmp = 0  # position x

# État réel de départ obtenu par un reset DÉTERMINISTE (graine fixe) : les deux
# trajectoires — imaginée et réelle — partent exactement du même état, sinon la
# comparaison n'a aucun sens.
SEED_CMP = 123
obs_start, _ = env.reset(seed=SEED_CMP)
s_start = torch.from_numpy(obs_start).float()

# Planifier une séquence d'actions avec le modèle final depuis cet état
_, mu_compare = planner_final.plan(
    s_start, dyn_final,
    lambda o, a, no: halfcheetah_reward(o, a, no, dt=env.unwrapped.dt),
    n_particles=P_cmp,
)
actions_cmp = mu_compare[:H_cmp]  # [H_cmp, act_dim]

# Trajectoire IMAGINÉE : moyenne des particules TS∞ depuis s_start
traj_imag = rollout_particles(dyn_final, s_start, actions_cmp, P_cmp, mode="tsinf", seed=0)
# [H_cmp+1, P_cmp, obs_dim]
mu_imag  = traj_imag[:, :, dim_cmp].mean(dim=1).numpy()   # [H_cmp+1]
std_imag = traj_imag[:, :, dim_cmp].std(dim=1).numpy()

# Trajectoire RÉELLE : mêmes actions, MÊME état de départ (reset à la même graine)
obs_r, _ = env.reset(seed=SEED_CMP)
x_real = [float(obs_r[dim_cmp])]
for t in range(H_cmp):
    obs_r, _, terminated, truncated, _ = env.step(actions_cmp[t].detach().numpy())
    x_real.append(float(obs_r[dim_cmp]))
    if terminated or truncated:
        break

n = len(x_real)
t_axis_cmp = np.arange(n)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_axis_cmp, mu_imag[:n], color="steelblue", lw=2, label="Imagination (moyenne particules)")
ax.fill_between(t_axis_cmp,
                mu_imag[:n] - std_imag[:n],
                mu_imag[:n] + std_imag[:n],
                alpha=0.25, color="steelblue", label=r"Imagination $\pm\sigma$")
ax.plot(t_axis_cmp, x_real, color="crimson", lw=2, ls="--", marker="o", ms=4, label="Réalité")
ax.set_xlabel("Pas de temps")
ax.set_ylabel(f"Dim {dim_cmp} (position x)")
ax.set_title("Imagination vs réalité : même état de départ, l'écart grandit avec l'horizon")
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Sur les premiers pas, l'imagination du modèle **colle** à la réalité (la trajectoire
vraie reste dans la bande des particules), puis l'écart se creuse à mesure que les erreurs se
composent et que la bande s'élargit. C'est précisément pourquoi MPC **replanifie à chaque pas** plutôt
que de dérouler tout l'horizon : on n'exécute que là où le modèle est encore fiable, et on se recale
sur l'état vrai avant que l'imagination ne dérive.

## Conclusion et ponts vers MBPO / Dreamer

### Ce qu'on a construit

Dans ce notebook, on a construit **PETS** comme une méthode model-based profonde qui ne cherche pas à
apprendre immédiatement une politique. À la place, elle apprend un modèle probabiliste de dynamique,
puis planifie en ligne à chaque pas réel.

La boucle complète est :

1. **Collecter** quelques transitions réelles ;
2. entraîner un **ensemble de réseaux probabilistes** ;
3. propager des **particules** dans ce modèle pour évaluer des plans ;
4. optimiser une séquence d'actions avec **CEM** ;
5. exécuter seulement la première action avec **MPC** ;
6. observer le vrai état suivant et recommencer.

| Brique | Rôle |
|--------|------|
| Réseau probabiliste | Prédit $\Delta s$ avec une moyenne et une variance |
| Tête de variance | Capture l'incertitude aléatoire propre à la transition |
| Deep ensemble | Capture l'incertitude épistémique par désaccord entre modèles |
| Bootstrap des membres | Rend les modèles suffisamment différents hors données |
| Trajectory sampling | Propage l'incertitude avec des particules |
| TS∞ | Fixe un membre par particule pour garder une hypothèse cohérente dans le temps |
| CEM | Cherche une bonne séquence d'actions par échantillonnage et sélection d'élites |
| MPC | N'exécute que la première action, puis replanifie depuis l'état réel |

### Ce que PETS gagne

PETS illustre une forme très efficace du model-based RL : **réfléchir beaucoup dans le modèle avant de
dépenser une action réelle**. Contrairement à DDPG, TD3 ou SAC, qui apprennent une politique par essais
successifs, PETS peut exploiter très tôt les transitions disponibles. Chaque nouvelle donnée améliore
le modèle, et chaque amélioration du modèle améliore immédiatement la planification.

Le point central est la gestion de l'incertitude :

$$
\operatorname{Var}(s')
=
\underbrace{\mathbb{E}_b[\sigma_b^2(s,a)]}_{\text{aléatoire}}
+
\underbrace{\operatorname{Var}_b(\mu_b(s,a))}_{\text{épistémique}}.
$$

La tête de variance dit : « même si je connais le monde, cette transition reste bruitée ». Le désaccord
entre membres dit : « je manque de données ici, plusieurs dynamiques plausibles existent encore ». Cette
séparation rend les plans moins naïfs : le planner voit non seulement un futur moyen, mais aussi la
dispersion des futurs possibles.

### Pourquoi les résultats du mini-training restent modestes

Le mini-training du notebook montre le mécanisme, mais ne prétend pas reproduire les scores complets
du papier PETS. HalfCheetah est un environnement difficile, et PETS devient coûteux dès qu'on augmente
le nombre de plans, particules, modèles et pas réels.

Les résultats doivent donc se lire ainsi :

- le modèle apprend : la NLL baisse et le désaccord épistémique diminue avec les données ;
- le planner produit parfois des trajectoires meilleures que le hasard ;
- les retours restent bruyants, car un petit budget CEM + peu d'itérations réelles ne suffisent pas à
  stabiliser une vraie allure de course ;
- pour obtenir des scores benchmark, il faut un budget beaucoup plus proche du papier : plus de données,
  plus de particules, un ensemble plus entraîné, et davantage d'itérations de CEM.

C'est une démonstration pédagogique de PETS, pas une campagne MuJoCo complète.

### Limites structurelles de PETS

| Limite | Conséquence |
|--------|-------------|
| Planning en ligne | Chaque action demande des milliers de simulations dans le modèle |
| Horizon limité | Les erreurs du modèle s'accumulent vite au-delà de quelques dizaines de pas |
| CEM sans gradient | Robuste et simple, mais coûteux en échantillons de plans |
| Récompense supposée connue | Si $r(s,a,s')$ est inconnue, il faut apprendre une tête récompense |
| Biais de modèle | Le planner peut exploiter des erreurs ou hallucinations du modèle |
| Pas de politique amortie | Après entraînement, l'agent doit encore planifier à chaque pas |

Le compromis est donc très clair : PETS est **sample-efficient**, mais pas **compute-efficient** au
moment d'agir.





## Pont vers MBPO

Le talon d'Achille de PETS est le **coût du planning en ligne**. À chaque pas, l'agent relance CEM,
évalue des centaines ou milliers de séquences, propage des particules, puis jette toutes les actions
sauf la première.

**MBPO** garde l'idée la plus solide de PETS — l'ensemble probabiliste — mais l'utilise autrement.
Au lieu de planifier en ligne, MBPO génère de **courts rollouts imaginés** depuis des états réels et
les ajoute au replay buffer d'un agent off-policy, typiquement SAC.

$$
\underbrace{\text{PETS}}_{\substack{\text{modèle d'ensemble} \\ \text{planning MPC/CEM}}}
\quad\longrightarrow\quad
\underbrace{\text{MBPO}}_{\substack{\text{modèle d'ensemble} \\ \text{rollouts courts} \\ \text{SAC off-policy}}}
\quad\longrightarrow\quad
\underbrace{\text{Dreamer}}_{\substack{\text{monde latent} \\ \text{acteur-critique imaginé}}}
$$

La logique est simple : garder l'efficacité-échantillon du modèle, mais amortir le contrôle dans une
politique qui répond instantanément. Les rollouts imaginés restent courts pour limiter l'erreur
composée, exactement la faiblesse que PETS révèle.

### Ce que vous devriez pouvoir réexpliquer

Si le chapitre est clair, vous devriez pouvoir répondre sans notes à :

1. Pourquoi une MSE ne suffit pas pour apprendre une variance honnête.
2. Ce que signifie “bruit hétéroscédastique”.
3. Comment la variance totale se sépare entre aléatoire et épistémique.
4. Pourquoi le désaccord d'un ensemble mesure l'incertitude épistémique.
5. Pourquoi TS∞ fixe un modèle par particule sur tout l'horizon.
6. Comment CEM resserre une distribution sur les séquences d'actions.
7. Pourquoi MPC exécute seulement la première action.
8. Pourquoi PETS est sample-efficient mais coûteux au test.
9. Pourquoi MBPO remplace le planning en ligne par une politique entraînée sur rollouts courts.

## Références

- Chua, K., Calandra, R., McAllister, R. & Levine, S. (2018). *Deep Reinforcement Learning in a Handful of Trials using Probabilistic Dynamics Models*. NeurIPS. [Proceedings](https://proceedings.neurips.cc/paper/2018/hash/3de568f8597b94bda53149c7d7f5958c-Abstract.html) / [arXiv:1805.12114](https://arxiv.org/abs/1805.12114).  
  **Référence centrale du notebook** : PETS, ensembles probabilistes, trajectory sampling, TS1/TS∞, MPC avec CEM.

- Lakshminarayanan, B., Pritzel, A. & Blundell, C. (2017). *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles*. NeurIPS. [arXiv:1612.01474](https://arxiv.org/abs/1612.01474).  
  Fondement pratique des **deep ensembles** : le désaccord entre membres sert d'approximation simple de l'incertitude épistémique.

- Kendall, A. & Gal, Y. (2017). *What Uncertainties Do We Need in Bayesian Deep Learning for Computer Vision?* NeurIPS. [arXiv:1703.04977](https://arxiv.org/abs/1703.04977).  
  Clarifie la distinction **aléatoire / épistémique** utilisée dans le notebook, même si l'article est orienté vision.

- Rubinstein, R. Y. (1999). *The Cross-Entropy Method for Combinatorial and Continuous Optimization*. *Methodology and Computing in Applied Probability*. [Springer](https://link.springer.com/article/10.1023/A:1010091220143).  
  Source historique de la **méthode d'entropie croisée** : échantillonner, garder les élites, recentrer la distribution.

- De Boer, P.-T., Kroese, D. P., Mannor, S. & Rubinstein, R. Y. (2005). *A Tutorial on the Cross-Entropy Method*. *Annals of Operations Research*. [PDF](https://people.smp.uq.edu.au/DirkKroese/ps/aortut.pdf).  
  Tutoriel clair sur CEM, utile pour comprendre pourquoi l'optimiseur utilisé par PETS fonctionne sans gradient.

- Rawlings, J. B., Mayne, D. Q. & Diehl, M. (2017). *Model Predictive Control: Theory, Computation, and Design*. [Site du livre](https://sites.engineering.ucsb.edu/~jbraw/mpc/).  
  Référence de fond pour **MPC** : planifier sur un horizon, exécuter seulement la première action, puis replanifier.

- Todorov, E., Erez, T. & Tassa, Y. (2012). *MuJoCo: A physics engine for model-based control*. IROS. [IEEE](https://ieeexplore.ieee.org/document/6386109).  
  Référence du simulateur physique derrière HalfCheetah et les benchmarks MuJoCo.

- Gymnasium / Farama Foundation. *HalfCheetah-v5 documentation*. [Documentation](https://gymnasium.farama.org/environments/mujoco/half_cheetah/).  
  Détaille l'observation, l'action, la récompense et les conventions de l'environnement utilisé dans le notebook.

- Janner, M., Fu, J., Zhang, M. & Levine, S. (2019). *When to Trust Your Model: Model-Based Policy Optimization*. NeurIPS. [arXiv:1906.08253](https://arxiv.org/abs/1906.08253).  
  Pont direct vers le notebook suivant : MBPO garde le modèle d'ensemble, mais remplace le planning en ligne par des rollouts courts pour entraîner SAC.
